# 05 · CUB70: can natural photographs reveal warning signs of concept backwash?

**Main question.** On real bird photographs, does a named concept score
follow its mapped body region, or can bird identity and surrounding
context predict the answer when that region is not available?

**What changes from FunnyBird.** FunnyBird lets us replace one part while
holding the same scene fixed. CUB70 does not. This chapter therefore tests
weaker observational warning signs—mask visibility, score separation while
the mask is absent, matched species differences, and held-out predictive
organization. It never calls them a donor/source margin or a CUB backwash
rate.

**Model and population.** Accepted Standard non-RL CUB70 Koh Joint
ResNet-50 CBM, seed 1. RLv2 is not assumed necessary. The notebook first
asks whether the label/mask conflict that RLv2 would change is actually
present and connected to model behavior.

**Recall-proxy rule.** The old matched-recall metric is retained only after
a new calibration against FunnyBird's known controlled swaps. If that
calibration fails, recall remains a species-dependence health diagnostic
and is not presented as a backwash proxy.


## What this notebook can establish, and how it approaches the FunnyBird question

The broad research question is the same: does a concept score depend only on its
named region, or does surrounding species/body context help predict it? The
strongest FunnyBird result cannot be copied mechanically because CUB has no
renderer that replaces one part while holding the rest of the photograph fixed.
Therefore this notebook does **not** invent a CUB donor/source margin.

The CUB conclusion must instead be assembled from explicitly observational
predicates, in this order:

1. the available photograph, species, concept, and mask populations are known;
2. species/concept structure makes contextual shortcuts available;
3. positive labels and mapped-mask visibility sometimes disagree;
4. each exact concept output is healthy enough to interpret;
5. natural visibility of the mapped region changes raw concept `z`;
6. positive and negative labels remain separable in `z` when the mapped region
   is absent (`context_gap > 0`);
7. species still organizes `z` after exact concept and mask state are held fixed;
8. measured visibility, conflict, difficulty, support, and species account for
   some—but not necessarily all—held-out variation.

### A proxy must earn its name before it reaches CUB

The old recall analysis compares the accepted CBM's own positive recall between
two species after requiring both species to contain enough positive **and**
negative images for the same exact concept. Notebook 05 now tests that idea in
two stages:

1. **FunnyBird calibration in the methods appendix.** For each exact concept,
   compare its ordinary-image matched recall gap with its known controlled-swap
   event rate. The event rate is available only because FunnyBird can replace a
   part in the same scene.
2. **CUB70 application.** If the association is positive overall, positive
   after subtracting each part's average, and positive in at least four of five
   leave-one-part-out checks, call recall a **provisional ordinal warning
   proxy**. Otherwise call it `METHOD NOT CALIBRATED AS A BACKWASH PROXY` and
   retain it only as a species-dependence/model-health diagnostic.

Even a successful calibration cannot turn a CUB recall gap into a CUB
backwash rate. It can only say that one ordinary-image warning sign tracks the
controlled FunnyBird outcome well enough to inspect on CUB.

### The same three contributors, with CUB-valid substitutions

1. **visibility/occlusion** uses the released mapped mask, its area, and
   bilateral alternatives; this is a natural-image comparison, not a swap;
2. **label–visibility conflict** is the positive-label/mapped-mask-absent rate
   for every exact concept, retaining coverage counts; and
3. **exact-value difficulty** uses raw-logit health, recall, exact-value support,
   number of alternatives, and species support.

Species/body context is then tested as the remaining observational organizer.
The recall analysis restores the original standard-CBM CUB question while
borrowing only the matching and bootstrap refinements from `mcbm_recallv4`;
no MCBM numerical result is imported here.

| FunnyBird scientific question | CUB operation | Figure(s) | Claim boundary |
|---|---|---|---|
| Are the data/model outputs usable? | inventory, masks, conflict, raw-`z` health | 1–4 | same health question |
| Is species context available? | label structure and held-out species decoding | 2, 4b | availability, not causal use |
| Do named-region pixels matter? | compare naturally visible and hidden positive-labelled photographs | 5, 7 | weaker than a same-image swap |
| Does context retain concept information? | hidden-positive minus hidden-negative raw `z` | 6 | contextual prediction, not donor/source backwash |
| Are scores species-dependent? | matched recall/raw-`z` gaps and within-concept species residuals | 8, 10 | observational species association |
| What proposed contributors organize the result? | concept- and row-level held-out accounting | 9, 11 | prediction, not causal subtraction |
| Could masks be misleading? | rule-selected photographs with all masks | 12 | separates true occlusion from annotation limits |
| Does the pattern survive a larger species population? | deferred until the official full-CUB Koh result exists | later chapter | unavailable here |

### CUB capabilities and drawbacks used in the design

CUB provides 112 exact labelled concepts, species labels, real photographs, and
11 released anatomical masks. It permits raw-score health, natural visibility,
area, bilateral-mask, recall, species, and support analyses. Its masks are
coarser than many named attributes and can be absent even when a human can see
the region. It has no accepted clean deletion or donor swap. Consequently:

- `visibility_effect` asks whether visible positive-labelled photographs score
  differently from hidden positive-labelled photographs;
- `context_gap` asks whether context distinguishes positive from negative labels
  when the mapped region is absent;
- neither quantity is the FunnyBird final margin;
- photographs and mask examples must be inspected before interpreting extremes;
- converging results support context-dependent prediction, but only FunnyBird's
  controlled swap establishes the exact backwash event.

### Why RLv2 is not assumed at the start

RLv2 changes positive labels when a part is not visible. It tests one proposed
cause: label/mask conflict. CUB70 is first analyzed with its accepted Standard
labels. RLv2 becomes a justified future experiment only if the mask audit is
credible and the affected concepts also show model behavior consistent with
context dependence. The current masks are evaluation evidence, not yet an
accepted training-time relabeling source.

### Predictions stated before the results

- Healthy outputs should have nonzero raw-`z` spread, positive label separation,
  and above-chance balanced accuracy/recall.
- If local visibility helps, `visibility_effect` should usually be positive and
  larger visible areas should usually accompany higher `z`.
- If context predicts the concept without the mapped region, `context_gap`
  should remain positive and species should explain held-out variation within
  exact concept and mask state.
- If conflict/support/number of alternatives are sufficient explanations, adding
  them should lower held-out concept-level error. If not, a residual remains.
- Mixed or negative visibility effects must be investigated as pose, mask
  quality, collapse, or composition before being called evidence for backwash.


## Model and symbols — the complete minimum needed below

- **Model:** accepted seed-1 **ResNet-50 Koh-architecture Joint CBM** trained
  on CUB70. It is not the `minimal_cbm` CBM and not an MCBM.
- **Path:** image → 112 named raw concept scores → one linear 112-to-70
  species head. The species head reads the raw scores, not rounded yes/no
  answers.
- **Training loss:** `L_task + 0.01 * L_concept`, normalized in the same Koh
  Joint form used by the accepted training job.
- **Dimensions:** 70 bird species and the canonical 112 CUB concepts.

```text
image x_i
   |
   v
ResNet-50 image encoder
   |
   v
112 raw concept logits z_i
   |                    |
   |                    +--> sigmoid only for thresholded concept metrics
   |
   +--> one linear 112-to-70 species head --> species logits
```

| Symbol | Plain meaning | Used for |
|---|---|---|
| `c_ij` | processed 0/1 label for image `i`, exact concept `j` | supervision and label matching |
| `z_ij` | raw score emitted for exact concept `j` | primary model quantity |
| `p_ij=sigmoid(z_ij)` | bounded concept probability | thresholded performance only |
| `c_hat_ij=1[z_ij>0]` | the model's yes/no concept answer | recall and balanced accuracy |
| `v_ig` | whether the released mask for anatomical group `g` is present above the declared area threshold | observational visibility |
| `a_ig` | released-mask area divided by image area | observational size |

There is no MCBM latent `h` and no learned `q(h)` in this model. The raw score
`z` is produced directly by the Koh concept head and is also what the linear
species head reads. Ordinary accuracy checks whether an answer matches a
label. It does not reveal which pixels produced the answer.


In [ ]:
import os, sys, json, hashlib, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

CURATED=Path(os.environ["CURATED_DATA"]); CWD=Path.cwd()
REPO=CWD if (CWD/"analysis").is_dir() else CWD.parent
sys.path.insert(0,str(REPO/"analysis"))
sys.path.insert(0,str(REPO/"data"/"cub70"))
from cub70_parts import CUB70_PARTS, ATTRIBUTE_TYPE_TO_MASK, COARSE_TO_CUB70
from relabel_cub_with_cub70 import coarse_visibility
from matched_recall_proxy import (matched_species_diagnostics, matched_species_eligibility,
    funnybird_swap_targets, calibrate_recall_warning)
COLORS={"head":"#56B4E9","eye":"#CC79A7","beak":"#E69F00","neck":"#009E73",
        "body":"#0072B2","wing":"#D55E00","leg":"#777777","tail":"#F0E442"}
COARSE_ORDER=["head","eye","beak","neck","body","wing","leg","tail"]
COLLAPSE_TOL=1e-8
pd.set_option("display.max_rows", 250)
pd.set_option("display.max_columns", 40)

def require(path,command):
    path=Path(path)
    if not path.exists(): raise FileNotFoundError(f"Missing {path}\nProduce it with: {command}")
    return path
def family(name): return str(name).split("::",1)[0]
def add_mapping(E):
    E=E.copy(); E["attribute_type"]=E.concept_name.map(family)
    E["mask_group"]=E.attribute_type.map(ATTRIBUTE_TYPE_TO_MASK); return E
def attach(E,V):
    local=add_mapping(E); V=V.rename(columns={"image_name":"image","coarse":"mask_group"})
    return local[local.mask_group.notna()].merge(
        V[["image","mask_group","pixel_count","area_frac","visible"]],
        on=["image","mask_group"],how="inner",validate="many_to_one")
def balanced_accuracy(y,pred):
    y=np.asarray(y).astype(int); pred=np.asarray(pred).astype(int)
    tpr=(pred[y==1]==1).mean() if (y==1).any() else np.nan
    tnr=(pred[y==0]==0).mean() if (y==0).any() else np.nan
    return np.nanmean([tpr,tnr])

CUB70_MODEL_ROOT=CURATED/"koh_joint_resnet_v1"/"cub70"/"standard"/"seed1"
CUB70_MANIFEST=require(CUB70_MODEL_ROOT/"SUCCESS.json","complete accepted official Koh CUB70 seed 1")
E70P=require(CUB70_MODEL_ROOT/"final_test.parquet","complete accepted official Koh CUB70 evaluation")
FB_MODEL_ROOT=CURATED/"koh_joint_resnet_accelerated_converged_v1"/"funnybirds"/"standard"/"seed1"
FB_SWAP_ROOT=CURATED/"swap_koh_joint_resnet_accelerated_converged_v1_seed1"
FB_MODEL_MANIFEST=require(FB_MODEL_ROOT/"SUCCESS.json","complete accepted FunnyBird Standard convergence")
FB_SWAP_MANIFEST=require(FB_SWAP_ROOT/"SUCCESS.json","complete accepted FunnyBird controlled swaps")
for manifest_path in [CUB70_MANIFEST,FB_MODEL_MANIFEST,FB_SWAP_MANIFEST]:
    subprocess.run([sys.executable,str(REPO/"analysis"/"canonical_manifest.py"),
                    "verify","--manifest",str(manifest_path)],check=True)
cub70_meta=json.loads(CUB70_MANIFEST.read_text()).get("metadata",{})
expected={"framework":"koh_joint","backbone":"resnet50","dataset":"cub70",
          "labels":"standard","seed":"1"}
wrong={key:(cub70_meta.get(key),value) for key,value in expected.items()
       if str(cub70_meta.get(key))!=value}
if wrong: raise RuntimeError(f"official CUB70 manifest identity mismatch: {wrong}")
if "minimal_cbm" in str(E70P): raise RuntimeError("Notebook 05 rejected a minimal_cbm export")

VIS=require(CURATED/"cub70_visibility.parquet","bash data/cub70/prepare_all.sh")
RAWVIS=pd.read_parquet(VIS); V=coarse_visibility(RAWVIS,threshold=.001)
E70=add_mapping(pd.read_parquet(E70P))
required_e70={"image","y_true","y_pred","concept_index","concept_name","z","prob","gt_label"}
if required_e70-set(E70.columns):
    raise RuntimeError(f"official Koh CUB70 export missing {sorted(required_e70-set(E70.columns))}")
E70["image_export_record"]=E70.image.astype(str)
E70["image"]=E70.image_export_record.map(lambda value:Path(value).stem)
if E70.duplicated(["image","concept_index"]).any():
    raise RuntimeError("normalizing official Koh CUB70 image paths produced duplicate image/concept rows")
concept_order=(E70[["concept_index","concept_name"]].drop_duplicates()
               .sort_values("concept_index"))
if concept_order.concept_index.tolist()!=list(range(112)):
    raise RuntimeError("official Koh CUB70 concept indices are not exactly 0..111")
selection_path=require(CURATED/"CUB_processed"/"class_attr_data_10_cub70_original"/"selection_indices.json",
                       "rerun the canonical CUB70 data preparation")
selected_indices=json.loads(selection_path.read_text())
if len(selected_indices)!=112: raise RuntimeError("CUB70 selection_indices.json does not contain 112 entries")
attribute_by_concept={index:int(source_index)+1 for index,source_index in enumerate(selected_indices)}
E70["attribute_id"]=E70.concept_index.map(attribute_by_concept)
if E70.attribute_id.isna().any(): raise RuntimeError("could not map every Koh concept index to a CUB attribute ID")
J70=attach(E70,V)

FB_EVAL=pd.read_parquet(require(FB_MODEL_ROOT/"final_test.parquet","complete accepted FunnyBird Standard evaluation"))
FB_SWAPS=pd.read_csv(require(FB_SWAP_ROOT/"funnybirds-cbm-s1.csv","complete accepted FunnyBird controlled swaps"))
if "response_delta" not in FB_SWAPS:
    FB_SWAPS["response_delta"]=FB_SWAPS.margin-(FB_SWAPS.z_new_orig-FB_SWAPS.z_old_orig)
if len(FB_EVAL)!=FB_EVAL.image.nunique()*26:
    raise RuntimeError("FunnyBird calibration export is not one row per image and concept")
if len(FB_SWAPS)!=5000:
    raise RuntimeError("FunnyBird calibration requires all 5,000 accepted swaps")
# The accepted FunnyBird final-test export has ten images per species.
# Audit 1/2/3-row thresholds before analysis.  Two rows in each label
# class is the fixed small-population rule; three removed the complete
# estimand and was an implementation error discovered on first execution.
FB_SUPPORT_AUDIT=matched_species_eligibility(FB_EVAL,thresholds=(1,2,3))
display(FB_SUPPORT_AUDIT)
FB_RECALL_PAIRS,FB_RECALL_SUMMARY,FB_RECALL_ELIGIBILITY=matched_species_diagnostics(
    FB_EVAL,min_each=2,max_pairs_per_concept=50,bootstrap_repeats=200,seed=20260910)
if FB_RECALL_PAIRS.empty:
    raise RuntimeError(
        "FunnyBird positive-and-negative species matching produced no calibration pairs "
        "at the fixed two-per-label rule; see FB_SUPPORT_AUDIT above"
    )
FB_SWAP_TARGETS=funnybird_swap_targets(FB_SWAPS)
FB_CALIBRATION,FB_CALIBRATION_CHECKS,FB_PROXY_VERDICT=calibrate_recall_warning(
    FB_RECALL_SUMMARY,FB_SWAP_TARGETS)
fb_prevalence=(FB_EVAL.groupby(["concept_name","y_true"]).gt_label.mean())
print("FunnyBird recall calibration label population: accepted Standard final-test processed labels")
print("FunnyBird recall calibration rule: final-test species must each have >=2 positive and >=2 negative rows; no all-positive fallback")
print("FunnyBird eligible calibration concepts:",FB_RECALL_SUMMARY.concept_name.nunique(),
      "pairs:",len(FB_RECALL_PAIRS),"maximum species/concept prevalence:",float(fb_prevalence.max()))
print("FunnyBird recall-proxy verdict:",FB_PROXY_VERDICT)
identity_error=float(np.nanmax(np.abs(E70.prob.to_numpy()-1/(1+np.exp(-E70.z.clip(-50,50).to_numpy())))))
if identity_error>1e-5: raise RuntimeError(f"exported z is not the concept logit: max probability mismatch={identity_error}")
print(f"[EXPORTED RAW-LOGIT PASS] max |prob-sigmoid(z)|={identity_error:.3g}")
print("framework: Koh Joint; backbone: ResNet-50; minimal_cbm: rejected")
print("official CUB70 manifest:",CUB70_MANIFEST)
print("official CUB70 evaluation:",E70P)
print("CUB70 rows:",len(E70),"images:",E70.image.nunique(),"species:",E70.y_true.nunique(),"concepts:",E70.concept_name.nunique())
print("mask-matched images:",J70.image.nunique(),"fine masks:",sorted(RAWVIS.part.unique()))


## 1 · What population and mask evidence are available?

**Question.** What population and mask evidence are available?

**Variables and prediction.** Count prediction images, mask-matched images, species, exact concepts, 11 released masks, and eight coarse groups. Coverage losses must be explicit before any visible-versus-hidden comparison.

**Method.** Report fine-mask visibility and bilateral left/right support without inventing left/right concepts. A mask is visible when area/image area is at least 0.001; use all successfully joined photographs and print that denominator from the current official-Koh export.

**Inputs and model.** Accepted official Koh Joint ResNet-50 CUB70 Standard seed-1 final-test export, plus released CUB masks or original image-level labels when the figure names them. No diagnostic is trained and no model score is interpreted; this is an input/mask inventory.

**Numerical record.** The following code cell prints the complete table behind
the picture, including per-part/per-value denominators and exclusions. The plot
is a visual summary of that table, not a second hidden calculation.

### Figure 1 · What population and mask evidence are available?

**How to read the figure.** Panel A shows, for each of the 11 released CUB masks, the fraction of all
successfully joined photographs where mask area is at least 0.001 of image
area, the declared visibility threshold. The current joined-image
denominator is printed immediately above the plot rather than copied from
an older render.
Panel B shows the median visible mask area divided by image area. `leg` is the
CUB name; `foot` is never used here. Low coverage can mean true occlusion,
pose, or missing/coarse annotation, which later photographs must distinguish.
Visibility 0.25 means the released mask passes the threshold in 25% of the
joined photographs.


In [ ]:
# ALT: CUB70 inventory with visibility rates and median area for all 11 released part masks.
inventory=pd.DataFrame([
    {"population":"CUB70 prediction export","images":E70.image.nunique(),"species":E70.y_true.nunique(),"concepts":E70.concept_name.nunique()},
    {"population":"mask-matched CUB70","images":J70.image.nunique(),"species":J70.y_true.nunique(),"concepts":J70.concept_name.nunique()},
])
fine=RAWVIS.groupby("part").agg(images=("image_name","nunique"),visible_rate=("visible","mean"),median_area=("area_frac","median")).reindex(CUB70_PARTS)
mask_map=[]
for group in COARSE_ORDER:
    families=sorted(k for k,v in ATTRIBUTE_TYPE_TO_MASK.items() if v==group)
    mask_map.append({"analysis_group":group,
                     "released_mask_sources":", ".join(COARSE_TO_CUB70[group]),
                     "mapped_attribute_families":", ".join(families)})
MASK_MAP=pd.DataFrame(mask_map)
display(inventory); display(fine.round(4)); display(MASK_MAP)
fig,axes=plt.subplots(1,2,figsize=(13,4.5))
axes[0].bar(fine.index,fine.visible_rate,color="#0072B2"); axes[0].tick_params(axis="x",rotation=55)
axes[0].set_ylabel("fraction of images with visible mask"); axes[0].set_title("A · Visibility of all 11 released masks")
axes[1].bar(fine.index,fine.median_area,color="#E69F00"); axes[1].tick_params(axis="x",rotation=55)
axes[1].set_ylabel("median mask area / image area"); axes[1].set_title("B · Visible-region size")
fig.suptitle("Figure 1 · CUB70 mask population and coverage")
plt.tight_layout(); plt.show()


### First-pass review slot for Figure 1

This slot is intentionally **INCOMPLETE** until the figure above has executed
from the accepted Koh Joint manifest and has been displayed in chat.

- **Literal observation:** Fill from the rendered axes and printed denominators;
  do not copy values from the superseded `minimal_cbm` report.
- **Strongest alternative to test:** An absent released mask can mean missing/coarse annotation rather than physical occlusion.
- **Discriminating test:** Inspect bilateral counts, areas, and real images with all masks overlaid.
- **Verdict:** choose `KEEP`, `REVISE`, `REMOVE`, or `MISSING EVIDENCE` only
  after visual inspection.
- **Limited conclusion:** state only what this output directly supports.
- **Next question if this step is interpretable:** Is a species/concept shortcut available before looking at model behavior?


## 2 · Is species–concept structure available before model behavior?

**Question.** Is species–concept structure available before model behavior?

**Variables and prediction.** For each exact selected concept, count supporting species, positive images, and the number of alternatives in its attribute type. Uneven support and species association make contextual prediction possible but do not prove model use.

**Method.** Use labels only; no model score appears in this figure.

**Inputs and model.** Accepted official Koh Joint ResNet-50 CUB70 Standard seed-1 final-test export, plus released CUB masks or original image-level labels when the figure names them. No diagnostic is trained and no model score is used; this is label structure only.

**Numerical record.** The following code cell prints the complete table behind
the picture, including per-part/per-value denominators and exclusions. The plot
is a visual summary of that table, not a second hidden calculation.

### Figure 2 · Is species–concept structure available before model behavior?

**How to read the figure.** Each horizontal row is one exact concept. The x-axis is the number of the 70
species carrying that exact value. Dot color is the number of positive
photographs (yellow means more; purple means fewer), and dot area is the number of alternative
values in the same attribute type (larger means more alternatives). A gray
outline means no released-mask mapping. Example: x=20 means 20 species carry
that value. This is label structure, not model behavior.


In [ ]:
# ALT: Named CUB70 label-only plot showing species support, positive-image support, and number of alternatives for every exact concept; outlined dots mark concepts without a released-mask mapping.
LABEL=(E70.groupby(["attribute_type","concept_name","y_true"]).gt_label.mean().reset_index())
support=(LABEL.assign(supports=lambda d:d.gt_label>=.5).groupby(["attribute_type","concept_name"])
         .agg(species_support=("supports","sum"),species_total=("y_true","nunique")).reset_index())
pos=E70.groupby(["attribute_type","concept_name"]).gt_label.agg(positive_images="sum",total_images="size").reset_index()
support=support.merge(pos); support["alternatives_in_type"]=support.groupby("attribute_type").concept_name.transform("nunique")
support["mask_group"]=support.attribute_type.map(ATTRIBUTE_TYPE_TO_MASK)
support=support.sort_values(["attribute_type","concept_name"]).reset_index(drop=True)
y=np.arange(len(support)); fig,ax=plt.subplots(figsize=(12,max(16,.24*len(support))))
color_value=np.log1p(support.positive_images)
sizes=28+18*support.alternatives_in_type
edge=np.where(support.mask_group.isna(),"#555555","white")
sc=ax.scatter(support.species_support,y,c=color_value,s=sizes,cmap="viridis",
              edgecolors=edge,linewidths=.8)
ax.set_yticks(y); ax.set_yticklabels(support.concept_name,fontsize=7); ax.invert_yaxis()
ax.set_xlabel("number of CUB70 species carrying this exact concept value")
cb=fig.colorbar(sc,ax=ax,pad=.01)
cb.set_label("number of positive photographs (log color scale)")
from matplotlib.lines import Line2D
size_values=sorted(set([int(support.alternatives_in_type.min()),
                        int(support.alternatives_in_type.median()),
                        int(support.alternatives_in_type.max())]))
handles=[Line2D([0],[0],marker="o",linestyle="",markerfacecolor="#888888",
                markeredgecolor="white",markersize=np.sqrt(28+18*n)/1.5,
                label=f"{n} alternatives") for n in size_values]
handles.append(Line2D([0],[0],marker="o",linestyle="",markerfacecolor="#888888",
                      markeredgecolor="#555555",label="no released-mask mapping"))
ax.legend(handles=handles,loc="lower right",fontsize=8,title="dot size / outline")
fig.suptitle("Figure 2 · Exact-concept structure before model behavior")
plt.tight_layout(); plt.show(); display(support.round(3))


### First-pass review slot for Figure 2

This slot is intentionally **INCOMPLETE** until the figure above has executed
from the accepted Koh Joint manifest and has been displayed in chat.

- **Literal observation:** Fill from the rendered axes and printed denominators;
  do not copy values from the superseded `minimal_cbm` report.
- **Strongest alternative to test:** Uneven label structure only makes a shortcut possible; it does not show model use.
- **Discriminating test:** Decode held-out species from the raw concept vector and individual part blocks.
- **Verdict:** choose `KEEP`, `REVISE`, `REMOVE`, or `MISSING EVIDENCE` only
  after visual inspection.
- **Limited conclusion:** state only what this output directly supports.
- **Next question if this step is interpretable:** Does the learned representation actually store species information?


## 3 · How often is a positive label paired with no visible mapped region?

**Question.** How often is a positive label paired with no visible mapped region?

**Variables and prediction.** For concept `j`, conflict is `P(v_ig=0 | c_ij=1)`. High conflict means training/evaluation labels can be predicted without visible named-region evidence; it does not prove model use.

**Method.** Plot every exact mask-testable concept at a named y-position with its denominator.

**Inputs and model.** Accepted official Koh Joint ResNet-50 CUB70 Standard seed-1 final-test export, plus released CUB masks or original image-level labels when the figure names them. No diagnostic is trained and no model score is used; this is a label/released-mask data audit.

**Numerical record.** The following code cell prints the complete table behind
the picture, including per-part/per-value denominators and exclusions. The plot
is a visual summary of that table, not a second hidden calculation.

### Figure 3 · How often is a positive label paired with no visible mapped region?

**How to read the figure.** Each named row is one exact concept. The x-value is a data fraction:
among images labelled positive for that concept, what fraction has no visible
mapped mask? It is not a predicted probability. A value of 0.8 means 80 of
every 100 positive-labelled examples lack a visible released mask.


In [ ]:
# ALT: Aligned named dot plot of positive-label/mask conflict rates and denominators for every testable CUB70 exact concept.
exact=[]
for (t,c),d in J70.groupby(["attribute_type","concept_name"]):
    pos=d[d.gt_label==1]; vis=pos[pos.visible]; hid=pos[~pos.visible]
    neg_hid=d[(d.gt_label==0)&(~d.visible)]
    exact.append({"attribute_type":t,"concept_name":c,"mask_group":d.mask_group.iloc[0],
                  "n_positive":len(pos),"n_visible":len(vis),"n_hidden":len(hid),
                  "label_mask_conflict":len(hid)/len(pos) if len(pos) else np.nan,
                  "z_visible":vis.z.mean() if len(vis) else np.nan,
                  "z_hidden":hid.z.mean() if len(hid) else np.nan,
                  "visibility_effect":vis.z.mean()-hid.z.mean() if len(vis) and len(hid) else np.nan,
                  "context_gap":hid.z.mean()-neg_hid.z.mean() if len(hid) and len(neg_hid) else np.nan,
                  "n_hidden_negative":len(neg_hid)})
# `support` carries a plotting-only mask_group column. Keep the
# row-level mask_group above instead of creating mask_group_x/y.
EXACT=pd.DataFrame(exact).merge(
    support.drop(columns=["mask_group"],errors="ignore"),
    on=["attribute_type","concept_name"],how="left"
)
EXACT=EXACT.sort_values(["attribute_type","concept_name"]).reset_index(drop=True)
y=np.arange(len(EXACT)); fig,ax=plt.subplots(figsize=(12,max(16,.24*len(EXACT))))
ax.scatter(EXACT.label_mask_conflict,y,c=EXACT.mask_group.map(COLORS).fillna("#BBBBBB"),s=24)
tick=[f"{r.concept_name}  (hidden/positive={int(r.n_hidden)}/{int(r.n_positive)})" for r in EXACT.itertuples()]
ax.set_yticks(y); ax.set_yticklabels(tick,fontsize=7); ax.invert_yaxis()
ax.set_xlim(-.02,1.02); ax.set_xlabel("fraction of positive labels with mapped mask absent")
ax.set_title("Figure 3 · Label/mask conflict for every exact testable concept")
plt.tight_layout(); plt.show(); display(EXACT[["concept_name","mask_group","n_positive","n_hidden","label_mask_conflict"]].round(3))


### First-pass review slot for Figure 3

This slot is intentionally **INCOMPLETE** until the figure above has executed
from the accepted Koh Joint manifest and has been displayed in chat.

- **Literal observation:** Fill from the rendered axes and printed denominators;
  do not copy values from the superseded `minimal_cbm` report.
- **Strongest alternative to test:** Figure 12 shows that mask absence sometimes occurs even when the anatomical region is visibly present, especially for neck and beak mappings.
- **Discriminating test:** Inspect real photographs and all masks, then treat v=0 as released-mask absence rather than guaranteed physical occlusion.
- **Verdict:** choose `KEEP`, `REVISE`, `REMOVE`, or `MISSING EVIDENCE` only
  after visual inspection.
- **Limited conclusion:** state only what this output directly supports.
- **Next question if this step is interpretable:** Do positive-labelled raw scores differ when the mapped mask is present?


## 4 · Did the standard CUB70 CBM produce usable exact-concept outputs?

**Question.** Did the standard CUB70 CBM produce usable exact-concept outputs?

**Variables and prediction.** For every concept, compute raw-score spread, label separation, balanced accuracy, and positive recall. Exact collapse means `Q95(z)-Q05(z) <= 1e-8`; rounded probabilities are not used to diagnose collapse.

**Method.** Evaluate all 112 outputs and mark mask-testable concepts separately.

**Inputs and model.** Accepted official Koh Joint ResNet-50 CUB70 Standard seed-1 final-test export, plus released CUB masks or original image-level labels when the figure names them. No new diagnostic is trained. The accepted CBM's raw z and its own z>0 concept decision are reused.

**Numerical record.** The following code cell prints the complete table behind
the picture, including per-part/per-value denominators and exclusions. The plot
is a visual summary of that table, not a second hidden calculation.

### Figure 4 · Did the standard CUB70 CBM produce usable exact-concept outputs?

**How to read the figure.** Each row is one of 112 exact concepts and the four aligned panels have the
same definitions as FunnyBird Figure 1: `spread=Q95(z)-Q05(z)`; `label
separation=median(z|c=1)-median(z|c=0)`; balanced accuracy averages positive
and negative recall; positive recall is `P(z>0|c=1)`. Zero spread within
`1e-8` is the declared collapse rule. Moving right is healthier for the last
three panels; spread only asks whether the output varies at all. For example,
label separation +2 means the positive-label median is two logit units above
the negative-label median.


In [ ]:
# ALT: Four aligned raw-score and thresholded-health plots for every CUB70 exact concept, with exact collapsed slots reported.
rows=[]
for (t,c),d in E70.groupby(["attribute_type","concept_name"]):
    pos=d[d.gt_label==1].z; neg=d[d.gt_label==0].z
    spread=np.quantile(d.z,.95)-np.quantile(d.z,.05)
    rows.append({"attribute_type":t,"concept_name":c,"mask_group":d.mask_group.iloc[0],
                 "spread":spread,"collapsed":spread<=COLLAPSE_TOL,
                 "label_separation":pos.median()-neg.median() if len(pos) and len(neg) else np.nan,
                 "balanced_accuracy":balanced_accuracy(d.gt_label,d.z>0),
                 "positive_recall":((pos>0).mean() if len(pos) else np.nan),
                 "n_positive":len(pos),"n_negative":len(neg)})
HEALTH=pd.DataFrame(rows).sort_values(["attribute_type","concept_name"]).reset_index(drop=True)
images=E70[["image","y_true","y_pred"]].drop_duplicates("image")
display(pd.DataFrame([{"images":len(images),"species":images.y_true.nunique(),
                      "task_accuracy":(images.y_true==images.y_pred).mean(),
                      "concept_accuracy":(E70.gt_label==E70.pred_label).mean()}]).round(4))
y=np.arange(len(HEALTH)); metrics=["spread","label_separation","balanced_accuracy","positive_recall"]
fig,axes=plt.subplots(1,4,figsize=(16,max(16,.24*len(HEALTH))),sharey=True)
colors=HEALTH.mask_group.map(COLORS).fillna("#BBBBBB")
for ax,m in zip(axes,metrics):
    ax.scatter(HEALTH[m],y,c=colors,s=17); ax.set_xlabel(m.replace("_"," "))
    if m=="label_separation": ax.axvline(0,color="black",lw=.8)
    if m in ["balanced_accuracy","positive_recall"]: ax.axvline(.5,color="gray",ls="--",lw=.8)
axes[0].set_yticks(y); axes[0].set_yticklabels(HEALTH.concept_name,fontsize=7); axes[0].invert_yaxis()
fig.suptitle("Figure 4 · Raw-score health guard for every exact CUB70 concept")
plt.tight_layout(); plt.show(); display(HEALTH[HEALTH.collapsed])
print("exact collapsed slots:",int(HEALTH.collapsed.sum()),"tolerance:",COLLAPSE_TOL)


### First-pass review slot for Figure 4

This slot is intentionally **INCOMPLETE** until the figure above has executed
from the accepted Koh Joint manifest and has been displayed in chat.

- **Literal observation:** Fill from the rendered axes and printed denominators;
  do not copy values from the superseded `minimal_cbm` report.
- **Strongest alternative to test:** Low or uneven performance can reflect the CUB70 training setup and label noise; it does not by itself establish grounding failure.
- **Discriminating test:** Keep the two collapsed slots out of positive grounding claims and analyze all remaining slots with raw z.
- **Verdict:** choose `KEEP`, `REVISE`, `REMOVE`, or `MISSING EVIDENCE` only
  after visual inspection.
- **Limited conclusion:** state only what this output directly supports.
- **Next question if this step is interpretable:** How often is a positive label paired with no released mapped mask?


## 4b · How much species identity is recoverable from the learned CUB70 concept vector?

**Question.** How much species identity is recoverable from the learned CUB70 concept vector?

**Variables and prediction.** On the same held-out split, decode species from each raw-logit block and from the corresponding processed 0/1 label block; also show the saved CBM's own task accuracy. A bar height is the fraction of held-out photographs whose species a separate diagnostic classifier guesses correctly. This measures recoverability from the supplied numbers, not grounding and not the saved CBM's task accuracy.

**Method.** Build one image-by-concept matrix and use a fixed stratified 70/30 split.

**Inputs and model.** Accepted official Koh Joint ResNet-50 CUB70 Standard seed-1 final-test export, plus released CUB masks or original image-level labels when the figure names them. Two new multinomial logistic-regression diagnostics are fitted after training: one receives known binary labels and one receives frozen raw z. The saved CBM species head is shown only as a separate reference line.

**Numerical record.** The following code cell prints the complete table behind
the picture, including per-part/per-value denominators and exclusions. The plot
is a visual summary of that table, not a second hidden calculation.

### Figure 4b · How much species identity is recoverable from the learned CUB70 concept vector?

**How to read the figure.** The y-axis is held-out species accuracy. Paired bars use learned raw logits
versus processed 0/1 concept labels on the same train/test split. The dashed
line is 1/70 blind chance; the dotted line is the saved CUB70 CBM's own task
accuracy. Unlike FunnyBird, `dimensions/70` is not a valid bucket baseline:
CUB region blocks contain multiple simultaneous attribute types and labels
vary within species. The label-only probe is therefore the valid structural
control. Raw-z accuracy above it is extra species information in the learned
representation, not proof of causal backwash.


### Before Figure 4b: what exactly are the grey and colored bars?

Each photograph has 112 processed binary labels `c`. A small portion of one
row might read `black bill=1`, `grey bill=0`, `striped tail=1`. This complete
row is the photograph's **attribute pattern**: its collection of known yes/no
concept answers after preprocessing.

We train two separate diagnostic classifiers after the CBM is finished:

- **grey bar:** the classifier receives known 0/1 labels `c`;
- **colored bar:** the classifier receives learned raw scores `z`;
- **bar height:** the fraction of held-out photographs whose species it guesses.

For example, grey `complete = 1.0` means this diagnostic classifier identified
every held-out species correctly from all 112 known yes/no answers. It does not
mean the saved CUB70 CBM has 100% task accuracy; that separate accuracy is the
dotted line.

“Species information” therefore means **species is statistically recoverable
from these numbers**. It does not identify the responsible pixels. A part block
can reveal species and still be well grounded, so later visibility/context tests
remain necessary.

> **IMPORTANT: Species leakage makes backwash possible, but leakage alone does
> not cause it. FunnyBird wing proves this: wing `z` reveals species while its
> controlled swaps remain strongly grounded. CUB has no equivalent controlled
> swap, so this figure cannot rank CUB grounding.**


In [ ]:
# ALT: Held-out CUB70 species-decoding accuracy from raw concept logits versus corresponding processed labels, with blind chance and saved-model task accuracy.
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
X=E70.pivot_table(index="image",columns="concept_name",values="z",aggfunc="first")
C=E70.pivot_table(index="image",columns="concept_name",values="gt_label",aggfunc="first").loc[X.index,X.columns]
image_rows=E70[["image","y_true","y_pred"]].drop_duplicates().set_index("image").loc[X.index]
y=image_rows.y_true
saved_task_accuracy=float((image_rows.y_true==image_rows.y_pred).mean())
tr,te=train_test_split(np.arange(len(X)),test_size=.30,random_state=20260803,stratify=y)
cmap=E70[["concept_name","mask_group"]].drop_duplicates().set_index("concept_name").mask_group
blocks={"complete z":list(X.columns)}
blocks.update({g:[c for c in X.columns if cmap.get(c)==g] for g in COARSE_ORDER})
rows=[]
for name,cols in blocks.items():
    if not cols: continue
    raw_model=make_pipeline(StandardScaler(),LogisticRegression(max_iter=4000,C=1.0,random_state=20260803))
    label_model=make_pipeline(StandardScaler(),LogisticRegression(max_iter=4000,C=1.0,random_state=20260803))
    raw_model.fit(X.iloc[tr][cols],y.iloc[tr]); label_model.fit(C.iloc[tr][cols],y.iloc[tr])
    rows.append({"block":name,
                 "raw_z_accuracy":accuracy_score(y.iloc[te],raw_model.predict(X.iloc[te][cols])),
                 "processed_label_accuracy":accuracy_score(y.iloc[te],label_model.predict(C.iloc[te][cols])),
                 "dimensions":len(cols)})
SPECIES_PROBE=pd.DataFrame(rows)
x=np.arange(len(SPECIES_PROBE)); w=.36; fig,ax=plt.subplots(figsize=(11,5))
ax.bar(x-w/2,SPECIES_PROBE.processed_label_accuracy,w,label="known 0/1 label probe",color="#BBBBBB")
ax.bar(x+w/2,SPECIES_PROBE.raw_z_accuracy,w,label="learned raw-z probe",
       color=["#333333"]+[COLORS.get(x,"#BBBBBB") for x in SPECIES_PROBE.block.iloc[1:]])
ax.set_xticks(x); ax.set_xticklabels(SPECIES_PROBE.block,rotation=30,ha="right")
ax.axhline(1/y.nunique(),color="black",ls="--",label="chance = 1/70")
ax.axhline(saved_task_accuracy,color="#D55E00",ls=":",label=f"saved CUB70 CBM task accuracy = {saved_task_accuracy:.3f}")
ax.set_ylim(0,1)
ax.set_ylabel("held-out species accuracy"); ax.set_title("Figure 4b · Species decoded from CUB70 raw concept logits")
ax.legend(); plt.tight_layout(); plt.show(); display(SPECIES_PROBE.round(3))


### First-pass review slot for Figure 4b

This slot is intentionally **INCOMPLETE** until the figure above has executed
from the accepted Koh Joint manifest and has been displayed in chat.

- **Literal observation:** Fill from the rendered axes and printed denominators;
  do not copy values from the superseded `minimal_cbm` report.
- **Strongest alternative to test:** High species accuracy from a block does not show whether that block uses its named pixels. It only shows that species can be recovered from the supplied numbers.
- **Discriminating test:** Compare this availability diagnostic with visibility_effect, context_gap, and the held-out row-level species contribution; CUB has no controlled grounding outcome.
- **Verdict:** choose `KEEP`, `REVISE`, `REMOVE`, or `MISSING EVIDENCE` only
  after visual inspection.
- **Limited conclusion:** state only what this output directly supports.
- **Next question if this step is interpretable:** Does natural visibility change the raw score of a positive-labelled concept?


## 5 · Does natural visibility change the raw score of a positive-labelled concept?

**Question.** Does natural visibility change the raw score of a positive-labelled concept?

**Variables and prediction.** `visibility_effect_j = mean(z|c=1,v=1)-mean(z|c=1,v=0)`. Positive values mean visible examples score higher; negative values require investigation rather than automatic backwash language.

**Method.** Require at least ten visible and ten hidden positive examples and show every eligible exact concept.

**Inputs and model.** Accepted official Koh Joint ResNet-50 CUB70 Standard seed-1 final-test export, plus released CUB masks or original image-level labels when the figure names them. No diagnostic is trained. Frozen raw z from the accepted CBM is grouped by natural released-mask state.

**Numerical record.** The following code cell prints the complete table behind
the picture, including per-part/per-value denominators and exclusions. The plot
is a visual summary of that table, not a second hidden calculation.

### Figure 5 · Does natural visibility change the raw score of a positive-labelled concept?

**How to read the figure.** Each named point is one exact concept. The x-axis is
`mean positive-labelled z when visible - mean positive-labelled z when hidden`.
Right of zero means visibility accompanies a higher raw score; left means the
visible group scores lower. Unlike a FunnyBird swap, these are different
photographs, so pose, species composition, and mask quality can also differ.


In [ ]:
# ALT: Zero-centered raw-logit visibility effects for every eligible CUB70 exact concept with visible and hidden counts.
VE=EXACT[(EXACT.n_visible>=10)&(EXACT.n_hidden>=10)&EXACT.visibility_effect.notna()].copy()
VE=VE.sort_values(["mask_group","attribute_type","concept_name"]).reset_index(drop=True)
y=np.arange(len(VE)); fig,ax=plt.subplots(figsize=(11,max(9,.23*len(VE))))
ax.scatter(VE.visibility_effect,y,c=VE.mask_group.map(COLORS).fillna("#BBBBBB"),s=30)
ax.axvline(0,color="black",lw=1); ax.set_yticks(y); ax.set_yticklabels(VE.concept_name,fontsize=6); ax.invert_yaxis()
ax.set_xlabel("visibility_effect in raw z units (visible − hidden)")
ax.set_title("Figure 5 · Natural-visibility effect for every eligible exact concept")
plt.tight_layout(); plt.show(); display(VE[["concept_name","mask_group","n_visible","n_hidden","z_hidden","z_visible","visibility_effect"]].round(3))


### First-pass review slot for Figure 5

This slot is intentionally **INCOMPLETE** until the figure above has executed
from the accepted Koh Joint manifest and has been displayed in chat.

- **Literal observation:** Fill from the rendered axes and printed denominators;
  do not copy values from the superseded `minimal_cbm` report.
- **Strongest alternative to test:** Visible and mask-absent photographs differ in species, pose, background, and mask quality; negative effects need not be inverse pixel use.
- **Discriminating test:** Test bilateral/area dose response, species matching, same-image model robustness, and real-image mask quality.
- **Verdict:** choose `KEEP`, `REVISE`, `REMOVE`, or `MISSING EVIDENCE` only
  after visual inspection.
- **Limited conclusion:** state only what this output directly supports.
- **Next question if this step is interpretable:** When the mapped mask is absent, does contextual label separation remain?


## 6 · Does contextual concept information remain when the named region is hidden?

**Question.** Does contextual concept information remain when the named region is hidden?

**Variables and prediction.** `context_gap_j = mean(z|c=1,v=0)-mean(z|c=0,v=0)`. A positive gap means outside-region information distinguishes the label while the mapped region is hidden; it is not a donor/source margin.

**Method.** Require at least ten hidden positives and ten hidden negatives.

**Inputs and model.** Accepted official Koh Joint ResNet-50 CUB70 Standard seed-1 final-test export, plus released CUB masks or original image-level labels when the figure names them. No diagnostic is trained. Frozen raw z from the accepted CBM is grouped by label and natural released-mask state.

**Numerical record.** The following code cell prints the complete table behind
the picture, including per-part/per-value denominators and exclusions. The plot
is a visual summary of that table, not a second hidden calculation.

### Figure 6 · Does contextual concept information remain when the named region is hidden?

**How to read the figure.** Each named point is one exact concept. The x-axis is the hidden-positive mean
raw score minus the hidden-negative mean raw score. A value of +4 means that,
even when the mapped region is absent, positive-labelled photographs score
four raw-logit units above negative-labelled photographs. That is contextual
prediction; it is not a donor/source margin and does not identify the cue.


In [ ]:
# ALT: Zero-centered raw-logit hidden-context gaps for every eligible CUB70 exact concept.
CG=EXACT[(EXACT.n_hidden>=10)&(EXACT.n_hidden_negative>=10)&EXACT.context_gap.notna()].copy()
CG=CG.sort_values(["mask_group","attribute_type","concept_name"]).reset_index(drop=True)
y=np.arange(len(CG)); fig,ax=plt.subplots(figsize=(11,max(9,.23*len(CG))))
ax.scatter(CG.context_gap,y,c=CG.mask_group.map(COLORS).fillna("#BBBBBB"),s=30)
ax.axvline(0,color="black",lw=1); ax.set_yticks(y); ax.set_yticklabels(CG.concept_name,fontsize=6); ax.invert_yaxis()
ax.set_xlabel("context_gap in raw z units (hidden positive − hidden negative)")
ax.set_title("Figure 6 · Hidden-region contextual separation")
plt.tight_layout(); plt.show(); display(CG[["concept_name","mask_group","n_hidden","n_hidden_negative","context_gap"]].round(3))


### First-pass review slot for Figure 6

This slot is intentionally **INCOMPLETE** until the figure above has executed
from the accepted Koh Joint manifest and has been displayed in chat.

- **Literal observation:** Fill from the rendered axes and printed denominators;
  do not copy values from the superseded `minimal_cbm` report.
- **Strongest alternative to test:** Released-mask absence is a noisy proxy: species, pose, background, annotation quality, and visibly present but unmasked regions can all create separation.
- **Discriminating test:** Match species support, center within exact concept/mask state, and inspect the selected photographs and masks.
- **Verdict:** choose `KEEP`, `REVISE`, `REMOVE`, or `MISSING EVIDENCE` only
  after visual inspection.
- **Limited conclusion:** state only what this output directly supports.
- **Next question if this step is interpretable:** Can bilateral visibility or region area explain the score patterns more simply?


## 7 · Do bilateral visibility and visible area offer simpler explanations?

**Question.** Do bilateral visibility and visible area offer simpler explanations?

**Variables and prediction.** For eye, wing, and leg, retain left/right masks and compare zero, one, or two visible sides. Separately estimate within-concept area dose response. A monotone increase supports local visual evidence; non-monotone patterns motivate pose or species controls.

**Method.** Use only positive-labelled rows and raw `z`.

**Inputs and model.** Accepted official Koh Joint ResNet-50 CUB70 Standard seed-1 final-test export, plus released CUB masks or original image-level labels when the figure names them. No diagnostic is trained. Frozen raw z is summarized by bilateral mask count and within-concept area quartile.

**Numerical record.** The following code cell prints the complete table behind
the picture, including per-part/per-value denominators and exclusions. The plot
is a visual summary of that table, not a second hidden calculation.

### Figure 7 · Do bilateral visibility and visible area offer simpler explanations?

**How to read the figure.** The bilateral panel compares mean raw `z` when zero, one, or two eye/wing/leg
masks are visible. The area panel asks, within the same exact concept, whether
larger visible masks accompany higher `z`. A steady upward pattern would fit
local pixel reliance; mixed directions leave pose, species, and annotation as
alternatives. The area outcome is
`area_effect_j = mean(z | largest visible-area quartile, c=1) -
mean(z | smallest visible-area quartile, c=1)`; +1 means the largest-area
positive images score one raw-logit unit higher. Colors identify CUB groups.


In [ ]:
# ALT: CUB70 raw-logit response by number of visible bilateral masks and by within-concept visible-area quartiles.
pairmap={"eye":["left_eye","right_eye"],"wing":["left_wing","right_wing"],"leg":["left_leg","right_leg"]}
side=[]
for group,parts2 in pairmap.items():
    d=RAWVIS[RAWVIS.part.isin(parts2)]
    pv=d.pivot(index="image_name",columns="part",values="visible").fillna(False)
    pa=d.pivot(index="image_name",columns="part",values="area_frac").fillna(0)
    for image in pv.index:
        side.append({"image":image,"mask_group":group,"visible_sides":int(pv.loc[image].sum()),"bilateral_area":float(pa.loc[image].sum())})
SIDE=pd.DataFrame(side)
B=J70[(J70.gt_label==1)&J70.mask_group.isin(pairmap)].merge(SIDE,on=["image","mask_group"])
BS=B.groupby(["mask_group","visible_sides"]).agg(n=("z","size"),mean_z=("z","mean")).reset_index()
dose=[]
for (t,c),d in J70[(J70.gt_label==1)&(J70.area_frac>0)].groupby(["attribute_type","concept_name"]):
    if len(d)<20 or d.area_frac.nunique()<4: continue
    q=pd.qcut(d.area_frac,4,duplicates="drop")
    if q.nunique()<2: continue
    lo=d.loc[q==q.cat.categories[0],"z"].mean(); hi=d.loc[q==q.cat.categories[-1],"z"].mean()
    dose.append({"attribute_type":t,"concept_name":c,"mask_group":d.mask_group.iloc[0],"area_effect":hi-lo,"n":len(d)})
DOSE=pd.DataFrame(dose)
fig,axes=plt.subplots(1,2,figsize=(13,4.5))
for g,d in BS.groupby("mask_group"): axes[0].plot(d.visible_sides,d.mean_z,"o-",label=g)
axes[0].set_xticks([0,1,2]); axes[0].set_xlabel("visible left/right masks"); axes[0].set_ylabel("mean raw z"); axes[0].legend()
for g,d in DOSE.groupby("mask_group"): axes[1].scatter([g]*len(d),d.area_effect,label=g,alpha=.65)
axes[1].axhline(0,color="black",lw=.8); axes[1].set_ylabel("largest-area quartile z − smallest-area quartile z")
fig.suptitle("Figure 7 · Bilateral visibility and area dose response")
plt.tight_layout(); plt.show(); display(BS.round(3)); display(DOSE.round(3))


### First-pass review slot for Figure 7

This slot is intentionally **INCOMPLETE** until the figure above has executed
from the accepted Koh Joint manifest and has been displayed in chat.

- **Literal observation:** Fill from the rendered axes and printed denominators;
  do not copy values from the superseded `minimal_cbm` report.
- **Strongest alternative to test:** Species and pose composition can overwhelm a natural-image area comparison, and small masks may be missing rather than physically absent.
- **Discriminating test:** Hold exact concept and species fixed and evaluate held-out row-level prediction.
- **Verdict:** choose `KEEP`, `REVISE`, `REMOVE`, or `MISSING EVIDENCE` only
  after visual inspection.
- **Limited conclusion:** state only what this output directly supports.
- **Next question if this step is interpretable:** Does performance differ by species after raw-label support is matched?


## 8 · Does concept performance differ between species after support is matched?

**Question.** Does concept performance differ between species after support is matched?

**Variables and prediction.** Join the original CUB per-image attribute labels to the official Koh raw `z` predictions. For each exact concept, compare species that each contain at least three positive and three negative images. Match both counts, then measure positive-recall, balanced-accuracy, and raw-score gaps. Persistent gaps support species-dependent behavior. They become a backwash warning only if the separate FunnyBird calibration in Appendix A passes its predeclared checks.

**Method.** Use the same tested formula and `matched_species_diagnostics` function for FunnyBird calibration and CUB70: original image-level labels, deterministic vectorized bootstrap, at most 50 species pairs per exact concept, and explicit alignment/eligibility counts. FunnyBird uses two rows per label because it has ten images per species; CUB70 uses three. No classifier is trained; `z>0` is the saved CBM's own concept decision.

**Inputs and model.** Accepted official Koh Joint ResNet-50 CUB70 Standard seed-1 final-test export, plus released CUB masks or original image-level labels when the figure names them. No diagnostic is trained. Recall and balanced accuracy reuse the accepted CBM's own z>0 rule; bootstrapping only resamples matched images.

**Numerical record.** The following code cell prints the complete table behind
the picture, including per-part/per-value denominators and exclusions. The plot
is a visual summary of that table, not a second hidden calculation.

### Figure 8 · Does concept performance differ between species after support is matched?

**How to read the figure.** A matched pair contains two species with enough raw positive and negative
examples for the same exact concept. Positive and negative counts are both
equalized and resampled with replacement. Panel A shows
`|positive_recall_A-positive_recall_B|`. Panel B shows the absolute gap in
balanced accuracy, where balanced accuracy is the average of positive and
negative recall. Panel C averages two raw-score gaps: the positive-image
mean difference and the negative-image mean difference. Zero means the two
matched species behave alike. A recall gap of 0.30 is a 30-percentage-point
difference; a raw-z gap of 2 is a two-logit-unit difference. No new
classifier is trained: every thresholded answer is the accepted CBM's own
`z>0` decision. These remain species-dependence/model-health diagnostics
unless Appendix A's FunnyBird calibration earns the weaker warning-proxy
label.


In [ ]:
# ALT: Three aligned CUB70 exact-concept plots using the saved Koh thresholded outputs and raw logits after matching positive and negative image counts between species.
cub_root=CURATED/"CUB_200_2011"
raw_candidates=[cub_root/"attributes"/"image_attribute_labels.txt",
                cub_root/"image_attribute_labels.txt"]
raw_path=next((p for p in raw_candidates if p.exists()),None)
images_path=cub_root/"images.txt"
if raw_path is None or not images_path.exists():
    raise FileNotFoundError(
        f"ERROR: raw CUB annotations missing under {cub_root}; need "
        "image_attribute_labels.txt and images.txt"
    )
raw=pd.read_csv(raw_path,sep=r"\s+",header=None,usecols=[0,1,2,3])
raw.columns=["image_id","attribute_id","raw_label","certainty"]
raw=raw[raw.certainty>=1].drop_duplicates(["image_id","attribute_id"])
image_rows=[]
for line in images_path.read_text().splitlines():
    image_id,relative=line.split(maxsplit=1)
    image_rows.append({"image_id":int(image_id),"image":Path(relative).stem})
image_ids=pd.DataFrame(image_rows)
raw_eval=(E70.merge(image_ids,on="image",how="left",validate="many_to_one")
          .merge(raw[["image_id","attribute_id","raw_label","certainty"]],
                 on=["image_id","attribute_id"],how="inner",validate="one_to_one"))
alignment_rate=len(raw_eval)/len(E70)
if alignment_rate<0.98:
    raise RuntimeError(
        f"ERROR: raw-label alignment covered only {alignment_rate:.1%} of E70 rows"
    )
RECALL,RS,ELIGIBILITY=matched_species_diagnostics(
    raw_eval,concept_col="concept_name",species_col="y_true",label_col="raw_label",
    score_col="z",min_each=3,max_pairs_per_concept=50,
    bootstrap_repeats=200,seed=20260910)
if RECALL.empty:
    raise RuntimeError(
        "ERROR: raw image-level CUB labels produced no eligible matched species pairs"
    )
RS=add_mapping(RS).sort_values(["attribute_type","concept_name"]).reset_index(drop=True)
fig,axes=plt.subplots(1,3,figsize=(19,max(12,.24*len(RS))),sharey=True)
y=np.arange(len(RS)); row_colors=RS.mask_group.map(COLORS).fillna("#BBBBBB")
axes[0].scatter(RS.mean_recall_gap,y,c=row_colors,s=24)
axes[1].scatter(RS.mean_balanced_accuracy_gap,y,c=row_colors,s=24)
axes[2].scatter(RS.mean_label_conditioned_raw_z_gap,y,c=row_colors,s=24)
axes[0].set_yticks(y); axes[0].set_yticklabels(RS.concept_name,fontsize=7); axes[0].invert_yaxis()
axes[0].set_xlabel("mean |positive recall A - B|")
axes[1].set_xlabel("mean |balanced accuracy A - B|")
axes[2].set_xlabel("mean label-conditioned raw-z gap")
fig.suptitle("Figure 8 · Species-matched differences in the saved CUB70 concept outputs")
plt.tight_layout(); plt.show()
display(pd.DataFrame([{"raw_alignment_rate":alignment_rate,"raw_rows":len(raw_eval),
                       "eligible_concepts":int((ELIGIBILITY.eligible_species>=2).sum()),
                       "matched_pairs":len(RECALL),"matching_rule":"both species >=3 positive and >=3 negative",
                       "new_diagnostic_trained":False,"funnybird_proxy_verdict":FB_PROXY_VERDICT}]).round(3))
display(RS.round(3))
display(RECALL.nlargest(25,"label_conditioned_raw_z_gap")[["concept_name","species_a","species_b",
    "matched_positive_n","matched_negative_n","recall_gap","balanced_accuracy_gap",
    "positive_raw_z_gap","label_conditioned_raw_z_gap"]].round(3))


### First-pass review slot for Figure 8

This slot is intentionally **INCOMPLETE** until the figure above has executed
from the accepted Koh Joint manifest and has been displayed in chat.

- **Literal observation:** Fill from the rendered axes and printed denominators;
  do not copy values from the superseded `minimal_cbm` report.
- **Strongest alternative to test:** Species still differ in pose, background, annotation certainty, and image quality; some matched supports are as small as three positives.
- **Discriminating test:** Replicate at the seed level and test species after exact concept and mask state with held-out images.
- **Verdict:** choose `KEEP`, `REVISE`, `REMOVE`, or `MISSING EVIDENCE` only
  after visual inspection.
- **Limited conclusion:** state only what this output directly supports.
- **Next question if this step is interpretable:** Do conflict, support, and alternatives organize the concept-level effects?


## 9 · Do conflict, support, and number of alternatives organize the exact-concept effects?

**Question.** Do conflict, support, and number of alternatives organize the exact-concept effects?

**Variables and prediction.** At the concept level, relate `visibility_effect` and `context_gap` to label/mask conflict, image support, species support, and alternatives in the attribute type. Held-out predictive improvement supports an organizing association, not a causal contribution.

**Method.** Use standardized numeric predictors and repeated five-fold ridge regression.

**Inputs and model.** Accepted official Koh Joint ResNet-50 CUB70 Standard seed-1 final-test export, plus released CUB masks or original image-level labels when the figure names them. A new post-hoc ridge predictor is fitted only to test held-out organization of the two already-computed observational outcomes. It does not alter the CBM.

**Numerical record.** The following code cell prints the complete table behind
the picture, including per-part/per-value denominators and exclusions. The plot
is a visual summary of that table, not a second hidden calculation.

### Figure 9 · Do conflict, support, and number of alternatives organize the exact-concept effects?

**How to read the figure.** The y-axis is held-out RMSE for predicting either the exact-concept visibility
effect or context gap; lower is better. Starting from an intercept, conflict,
image support, species support, and number of alternatives are added. A drop
means the added concept-level information generalizes; a rise supplies no
explanatory credit. RMSE 1.2 to 1.0 is improvement; 1.2 to 1.3 is not.
This does not subtract causal effects.


In [ ]:
# ALT: Cross-validated concept-level error after sequentially adding label conflict, image support, species support, and number of alternatives.
from sklearn.model_selection import RepeatedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
FEATURES=["label_mask_conflict","n_positive","species_support","alternatives_in_type"]
collapsed_names=set(HEALTH.loc[HEALTH.collapsed,"concept_name"])
ACCOUNT_BASE=EXACT[(EXACT.n_visible>=10)&(EXACT.n_hidden>=10)&(EXACT.n_hidden_negative>=10)
                   & ~EXACT.concept_name.isin(collapsed_names)].copy()
rows=[]
for outcome in ["visibility_effect","context_gap"]:
    d=ACCOUNT_BASE.dropna(subset=[outcome]).copy()
    cv=RepeatedKFold(n_splits=5,n_repeats=10,random_state=20260803)
    # The intercept-only comparison must obey the same held-out folds.
    # Using the full-data outcome mean would leak held-out outcomes into
    # the baseline and make every added feature look unfairly weak.
    baseline_mse=[]
    for train_index,test_index in cv.split(d):
        train_mean=float(d.iloc[train_index][outcome].mean())
        baseline_mse.extend((d.iloc[test_index][outcome]-train_mean)**2)
    baseline=float(np.sqrt(np.mean(baseline_mse)))
    for k in range(1,len(FEATURES)+1):
        model=make_pipeline(SimpleImputer(),StandardScaler(),Ridge(alpha=5.0))
        mse=-cross_val_score(model,d[FEATURES[:k]],d[outcome],cv=cv,scoring="neg_mean_squared_error")
        rows.append({"outcome":outcome,"stage":" + ".join(FEATURES[:k]),"rmse":float(np.sqrt(mse.mean())),"n_concepts":len(d)})
    rows.append({"outcome":outcome,"stage":"intercept only","rmse":baseline,"n_concepts":len(d)})
CONCEPT_ACCOUNT=pd.DataFrame(rows)
fig,axes=plt.subplots(1,2,figsize=(14,4.5))
for ax,outcome in zip(axes,["visibility_effect","context_gap"]):
    d=CONCEPT_ACCOUNT[CONCEPT_ACCOUNT.outcome==outcome]
    order=["intercept only"]+[" + ".join(FEATURES[:k]) for k in range(1,len(FEATURES)+1)]
    d=d.set_index("stage").reindex(order); ax.plot(range(len(d)),d.rmse,"o-")
    ax.set_xticks(range(len(d))); ax.set_xticklabels(["baseline","+ conflict","+ image support","+ species support","+ alternatives"],rotation=25,ha="right")
    ax.set_ylabel("cross-validated RMSE"); ax.set_title(outcome.replace("_"," "))
fig.suptitle("Figure 9 · Concept-level sequential observational accounting")
plt.tight_layout(); plt.show()
display(pd.DataFrame([{"shared_eligible_concepts":len(ACCOUNT_BASE),
    "excluded_collapsed":len(collapsed_names),"minimum_visible_positive":10,
    "minimum_hidden_positive":10,"minimum_hidden_negative":10}]))
display(CONCEPT_ACCOUNT.round(3))


### First-pass review slot for Figure 9

This slot is intentionally **INCOMPLETE** until the figure above has executed
from the accepted Koh Joint manifest and has been displayed in chat.

- **Literal observation:** Fill from the rendered axes and printed denominators;
  do not copy values from the superseded `minimal_cbm` report.
- **Strongest alternative to test:** Changing eligibility can change both baselines and apparent predictor gains, so the old numerical comparison cannot be carried forward.
- **Discriminating test:** Rerender this single corrected shared-population analysis, then compare the two outcomes.
- **Verdict:** choose `KEEP`, `REVISE`, `REMOVE`, or `MISSING EVIDENCE` only
  after visual inspection.
- **Limited conclusion:** state only what this output directly supports.
- **Next question if this step is interpretable:** Does species-dependent raw-z variation remain within concept and mask state?


## 10 · Does species explain raw-score variation within the same exact concept and visibility state?

**Question.** Does species explain raw-score variation within the same exact concept and visibility state?

**Variables and prediction.** First center `z` within each exact concept and visibility state, then summarize residual means by species. Persistent spread shows species-dependent contextual prediction beyond the current mask state.

**Method.** Require at least three rows for every displayed concept/state/species estimate.

**Inputs and model.** Accepted official Koh Joint ResNet-50 CUB70 Standard seed-1 final-test export, plus released CUB masks or original image-level labels when the figure names them. No diagnostic is trained. Means are subtracted within exact concept and mask state, then residuals are summarized by species.

**Numerical record.** The following code cell prints the complete table behind
the picture, including per-part/per-value denominators and exclusions. The plot
is a visual summary of that table, not a second hidden calculation.

### Figure 10 · Does species explain raw-score variation within the same exact concept and visibility state?

**How to read the figure.** First subtract the mean raw `z` for the same exact concept and visible/hidden
state. Each point then summarizes one species. Zero means the species matches
that controlled average; remaining spread means species still organizes the
score. Because photographs were not experimentally changed, this remains an
observational context effect. A residual of +3 means that species lies three
raw-logit units above the same concept-and-mask-state mean.


In [ ]:
# ALT: CUB70 species-level raw-logit residuals after centering within exact concept and visibility state for all eight coarse groups.
R=J70.copy(); R["concept_visibility_mean"]=R.groupby(["concept_name","visible"]).z.transform("mean")
R["z_after_concept_visibility"]=R.z-R.concept_visibility_mean
SP=(R.groupby(["mask_group","concept_name","visible","y_true"]).agg(n=("z","size"),residual=("z_after_concept_visibility","mean"))
      .reset_index().query("n>=3"))
fig,axes=plt.subplots(2,4,figsize=(16,8),sharey=True); axes=axes.ravel()
for ax,g in zip(axes,COARSE_ORDER):
    d=SP[SP.mask_group==g].sort_values("residual")
    ax.scatter(np.arange(len(d)),d.residual,s=12,color=COLORS[g],alpha=.7)
    ax.axhline(0,color="black",lw=.8); ax.set_title(f"{g}: {len(d)} estimates"); ax.set_xlabel("concept/state/species, sorted")
axes[0].set_ylabel("mean raw-z residual"); axes[4].set_ylabel("mean raw-z residual")
fig.suptitle("Figure 10 · Species variation after exact concept and mask state")
plt.tight_layout(); plt.show(); display(SP.groupby("mask_group").residual.agg(["min","median","max","std","count"]).round(3))


### First-pass review slot for Figure 10

This slot is intentionally **INCOMPLETE** until the figure above has executed
from the accepted Koh Joint manifest and has been displayed in chat.

- **Literal observation:** Fill from the rendered axes and printed denominators;
  do not copy values from the superseded `minimal_cbm` report.
- **Strongest alternative to test:** Small or uneven concept/state/species cells and correlated pose/background can produce extreme descriptive residuals.
- **Discriminating test:** Require held-out image prediction and shrunken estimates before giving species generalizing explanatory credit.
- **Verdict:** choose `KEEP`, `REVISE`, `REMOVE`, or `MISSING EVIDENCE` only
  after visual inspection.
- **Limited conclusion:** state only what this output directly supports.
- **Next question if this step is interpretable:** Does species reduce held-out row-level prediction error?


## 11 · What remains after row-level visibility and species are added sequentially?

**Question.** What remains after row-level visibility and species are added sequentially?

**Variables and prediction.** Predict raw `z` on stable held-out image folds: exact concept baseline, then mask visibility/area, then species. A reduction in held-out error shows organization by that block; remaining error is the residual, not proof of an unknown cause.

**Method.** Use training-fold shrunken group means and identical rows at every stage.

**Inputs and model.** Accepted official Koh Joint ResNet-50 CUB70 Standard seed-1 final-test export, plus released CUB masks or original image-level labels when the figure names them. New fold-specific group-mean prediction rules are estimated after the CBM is frozen. They predict raw z on held-out images and do not alter the CBM.

**Numerical record.** The following code cell prints the complete table behind
the picture, including per-part/per-value denominators and exclusions. The plot
is a visual summary of that table, not a second hidden calculation.

### Figure 11 · What remains after row-level visibility and species are added sequentially?

**How to read the figure.** The y-axis is held-out raw-`z` prediction error; lower is better. The same image
rows are used throughout. Start with exact concept identity, add mask
visibility and area, then add species. Each decrease measures extra predictive
organization on unseen images. The remaining nonzero error is the residual,
not automatically a new causal mechanism. RMSE 3.3 to 3.1 means the added
block improves unseen-image prediction by 0.2 logit units.


In [ ]:
# ALT: Held-out CUB70 raw-logit prediction error after sequentially adding visibility, area, and species to exact concept identity.
A=J70.copy()
# Fixed physical-area bins avoid selecting bin boundaries from held-out
# images. The first positive boundary is the visibility threshold.
A["area_bin"]=pd.cut(A.area_frac,[-np.inf,.001,.005,.02,.05,np.inf],
                      labels=False,include_lowest=True)
A["fold"]=A.image.map(lambda x:int(hashlib.sha1(str(x).encode()).hexdigest(),16)%5)
stages=[("exact concept",["concept_name"]),("+ visibility and area",["concept_name","visible","area_bin"]),
        ("+ species",["concept_name","visible","area_bin","y_true"])]
rows=[]
for stage,cols in stages:
    pred=pd.Series(index=A.index,dtype=float)
    for fold in range(5):
        tr=A[A.fold!=fold]; te=A[A.fold==fold]; prior=tr.z.mean()
        st=tr.groupby(cols).z.agg(["mean","count"]).reset_index(); st["estimate"]=(st["mean"]*st["count"]+prior*10)/(st["count"]+10)
        j=te[cols].merge(st[cols+["estimate"]],on=cols,how="left")
        pred.loc[te.index]=j.estimate.fillna(prior).to_numpy()
    rows.append({"stage":stage,"rmse":float(np.sqrt(np.mean((A.z-pred)**2))),"mae":float(np.mean(np.abs(A.z-pred)))})
ROW_ACCOUNT=pd.DataFrame(rows)
fig,ax=plt.subplots(figsize=(7,4)); ax.plot(ROW_ACCOUNT.stage,ROW_ACCOUNT.rmse,"o-",color="#0072B2")
ax.set_ylabel("held-out RMSE of raw z"); ax.set_title("Figure 11 · Row-level sequential observational accounting")
plt.tight_layout(); plt.show(); display(ROW_ACCOUNT.round(3))


### First-pass review slot for Figure 11

This slot is intentionally **INCOMPLETE** until the figure above has executed
from the accepted Koh Joint manifest and has been displayed in chat.

- **Literal observation:** Fill from the rendered axes and printed denominators;
  do not copy values from the superseded `minimal_cbm` report.
- **Strongest alternative to test:** Species can proxy pose, habitat, background, and collection effects, so predictive gain does not isolate a biological species-to-concept causal path.
- **Discriminating test:** A matched relabel/retrain or valid same-image intervention would be needed for a causal claim; neither is accepted for CUB yet.
- **Verdict:** choose `KEEP`, `REVISE`, `REMOVE`, or `MISSING EVIDENCE` only
  after visual inspection.
- **Limited conclusion:** state only what this output directly supports.
- **Next question if this step is interpretable:** Do real images show true occlusion, missing masks, or pose artifacts at the extremes?


## Textbook guide: the measurements are related questions, not interchangeable scores

The aligned figures deliberately put anatomical groups in the same row order,
but the panels do **not** all measure the same thing. FunnyBird has controlled
part replacement; CUB has natural photographs and released masks. We therefore
match the scientific question while naming the weaker CUB approximation.

| Scientific question | FunnyBird measurement | CUB measurement | Same operation? |
|---|---|---|---|
| Are labels present without visible part evidence? | renderer-derived label/visibility conflict | positive label with mapped mask absent | related; CUB masks are noisier |
| Is the concept output usable? | raw-`z` spread, balanced accuracy, positive recall | the same health checks | yes |
| Do named pixels affect the score? | controlled `response_delta` after donor insertion | visible-minus-hidden raw-`z` difference | no; CUB compares different photographs |
| Does context remain after local evidence is limited? | donorward response occurs but old source still wins | hidden positive-minus-negative raw-`z` gap | no; only FunnyBird has a donor/source margin |
| Does species still organize the score? | source-species residual after exact source/donor values | species residual after exact concept and mask state | related and observational |
| Is the exact inserted value recognized? | controlled post-swap value confusion | no clean equivalent | unavailable in CUB |

### Model health comes before grounding

For exact concept `j`, the model predicts positive when `z_ij>0`. Balanced
accuracy gives positive and negative examples equal weight:

`balanced_accuracy = (positive recall + negative recall) / 2`.

If 70% of positive examples and 80% of negative examples are correct, balanced
accuracy is `(0.70+0.80)/2 = 0.75`; the aligned summary plots ordinary concept
error `1-0.75 = 0.25`. A large error says the output is difficult. It does not
say whether the error came from context, weak pixels, or noisy labels.

An output is **collapsed** when its raw score is effectively constant across all
images: `Q95(z)-Q05(z) <= 1e-8`. For example, returning `z=+2.1` for every image
always predicts “present.” Positive recall would misleadingly equal 1, negative
recall would equal 0, and balanced accuracy would equal 0.5. Such an output did
not learn a usable image distinction and cannot support a grounding claim.

The health figure must name any collapsed outputs found in the **current
accepted model**.  Do not carry a collapsed-output name or count from another
checkpoint.  A collapsed output remains visible as a negative health result and
is excluded from positive grounding summaries; the exclusion table must print
its name, spread, positive recall, negative recall, and balanced accuracy.

### Direction of each CUB panel

| Panel type | A larger value means | Interpretation |
|---|---|---|
| **Data check: positive label / mask absent** | more positive labels lack a usable mapped mask | possible label/visibility conflict, but also possible missing annotation |
| **Health check: ordinary concept error** | worse positive/negative prediction | weak or difficult output; not automatically backwash |
| **Local evidence: visible - hidden raw `z`** | positive examples score higher when the region is visible | evidence that local pixels help; usually a good grounding sign |
| **Context evidence: hidden positive - negative raw `z`** | labels remain separated when the mapped mask is absent | context or unmeasured pixels remain informative |
| **Species context: residual spread** | species shift `z` after exact concept and mask state are centered | species-associated organization remains |

These quantities have different units and directions. They must not be added
into a synthetic “CUB backwash score.” Repeatedly unusual groups are stronger
observational candidates; only a controlled outcome can measure causal
backwash directly.


## 11a · Which exact CUB concepts carry each measured problem?

**Question.** Which exact CUB concepts carry each measured problem?

**Variables and prediction.** Align the same exact-concept rows across mask absence, ordinary concept error, visible-minus-hidden raw-z difference, hidden context gap, and within-concept species residual spread. If one anatomical family repeatedly contains the largest values, its coarse ranking reflects consistent exact concepts. Mixed rows show that coarse aggregation hides value-specific behavior.

**Method.** Retain all mask-testable exact concepts; leave unsupported measurements blank and show the exact denominators in the table.

**Inputs and model.** Accepted official Koh Joint ResNet-50 CUB70 Standard seed-1 final-test export, plus released CUB masks or original image-level labels when the figure names them. No new model is fitted; this aligns measurements already computed in earlier figures.

**Numerical record.** The following code cell prints the complete table behind
the picture, including per-part/per-value denominators and exclusions. The plot
is a visual summary of that table, not a second hidden calculation.

### Figure 11a · Which exact CUB concepts carry each measured problem?

**How to read the figure.** Every row is one exact mask-testable CUB concept, such as
`has_tail_pattern::striped`, and the row order is identical across all five
panels. Color identifies the coarse mask group. Panel A is the fraction of
positive labels with the mapped mask absent. Panel B is ordinary
classification error `1-balanced_accuracy`. Panel C is visible-minus-hidden
raw `z` among positive labels. Panel D is hidden-positive minus hidden-negative
raw `z`. Panel E is the standard deviation of species residuals after exact
concept and mask state are centered. Blank positions mean that exact concept
lacked the required visible/hidden/species support; they are not zeros.


In [ ]:
# ALT: Five aligned panels retaining every mask-testable exact CUB concept and showing unsupported quantities as missing rather than zero.
CUB_GROUP_ORDER=["tail","wing","beak","leg","eye","neck","body","head"]
collapsed_names=set(HEALTH.loc[HEALTH.collapsed,"concept_name"])
species_exact=(SP.groupby(["mask_group","concept_name"]).agg(
    species_residual_sd=("residual","std"),n_species_cells=("residual","size")).reset_index())
health_exact=HEALTH[["concept_name","balanced_accuracy","collapsed","n_positive","n_negative"]].copy()
health_exact=health_exact.rename(columns={"n_positive":"health_n_positive","n_negative":"health_n_negative"})
health_exact["classification_error"]=1-health_exact.balanced_accuracy
CUB_EXACT_SYN=(EXACT.merge(health_exact,on="concept_name",how="left")
    .merge(species_exact,on=["mask_group","concept_name"],how="left"))
CUB_EXACT_SYN.loc[CUB_EXACT_SYN.concept_name.isin(collapsed_names),
                  ["classification_error","visibility_effect","context_gap","species_residual_sd"]]=np.nan
CUB_EXACT_SYN.loc[(CUB_EXACT_SYN.health_n_positive<10)|(CUB_EXACT_SYN.health_n_negative<10),
                  "classification_error"]=np.nan
CUB_EXACT_SYN.loc[(CUB_EXACT_SYN.n_visible<10)|(CUB_EXACT_SYN.n_hidden<10),
                  "visibility_effect"]=np.nan
CUB_EXACT_SYN.loc[(CUB_EXACT_SYN.n_hidden<10)|(CUB_EXACT_SYN.n_hidden_negative<10),
                  "context_gap"]=np.nan
CUB_EXACT_SYN.loc[CUB_EXACT_SYN.n_species_cells<10,"species_residual_sd"]=np.nan
CUB_EXACT_SYN["group_order"]=CUB_EXACT_SYN.mask_group.map({g:i for i,g in enumerate(CUB_GROUP_ORDER)})
CUB_EXACT_SYN=CUB_EXACT_SYN.sort_values(
    ["group_order","context_gap","concept_name"],ascending=[True,False,True]).reset_index(drop=True)
y=np.arange(len(CUB_EXACT_SYN)); row_colors=CUB_EXACT_SYN.mask_group.map(COLORS).fillna("#888888")
panels=[
    ("label_mask_conflict","A · DATA CHECK: label / mask absent",(0,1)),
    ("classification_error","B · HEALTH CHECK: concept error",(0,1)),
    ("visibility_effect","C · LOCAL EVIDENCE: visible − hidden z",None),
    ("context_gap","D · CONTEXT: hidden positive − negative z",None),
    ("species_residual_sd","E · SPECIES CONTEXT: residual spread",None),
]
fig,axes=plt.subplots(1,5,figsize=(20,max(18,.225*len(CUB_EXACT_SYN))),sharey=True)
for ax,(column,title,limits) in zip(axes,panels):
    d=CUB_EXACT_SYN[column].notna()
    ax.scatter(CUB_EXACT_SYN.loc[d,column],y[d],c=row_colors[d],s=20)
    if column in ["visibility_effect","context_gap"]: ax.axvline(0,color="black",lw=.8)
    if limits: ax.set_xlim(*limits)
    ax.set_title(title,fontsize=10); ax.grid(axis="x",alpha=.2)
axes[0].set_yticks(y); axes[0].set_yticklabels(CUB_EXACT_SYN.concept_name,fontsize=6)
axes[0].invert_yaxis()
boundaries=CUB_EXACT_SYN.groupby("mask_group",sort=False).size().cumsum().iloc[:-1]-0.5
for ax in axes:
    for boundary in boundaries: ax.axhline(boundary,color="#BBBBBB",lw=.7)
fig.suptitle("Figure 11a · Exact CUB concepts aligned across measurement, health, context, and species questions", y=.998)
plt.tight_layout(rect=[0,0,1,.985]); plt.show()
display(CUB_EXACT_SYN[["mask_group","attribute_type","concept_name","n_positive","n_hidden",
    "label_mask_conflict","health_n_positive","health_n_negative","classification_error","n_visible","visibility_effect",
    "n_hidden_negative","context_gap","n_species_cells","species_residual_sd"]].round(3))


### First-pass review slot for Figure 11a

This slot is intentionally **INCOMPLETE** until the figure above has executed
from the accepted Koh Joint manifest and has been displayed in chat.

- **Literal observation:** Fill from the rendered axes and printed denominators;
  do not copy values from the superseded `minimal_cbm` report.
- **Strongest alternative to test:** A long aligned display can reveal where measurements coincide, but visual alignment alone does not establish that one quantity caused another.
- **Discriminating test:** Use the complete printed exact-concept table and the held-out tests rather than comparing only memorable rows.
- **Verdict:** choose `KEEP`, `REVISE`, `REMOVE`, or `MISSING EVIDENCE` only
  after visual inspection.
- **Limited conclusion:** state only what this output directly supports.
- **Next question if this step is interpretable:** Do the same measurements form a stable anatomical-group pattern?


## 11b · How are the available CUB contributors distributed across coarse anatomical groups?

**Question.** How are the available CUB contributors distributed across coarse anatomical groups?

**Variables and prediction.** Use the same anatomical order wherever possible and report five distinct quantities: positive-label/mask-absence rate, median exact-concept classification difficulty, median natural visibility effect, median hidden context gap, and species-residual spread. If CUB behaves like a simple diluted copy of FunnyBird, the same groups should repeatedly rank as difficult. If rankings differ, the contributors are distributed across concepts and cannot be reduced to one tail-to-wing grounding order.

**Method.** Do not sum the panels, do not call any panel a CUB donor/source margin, and print eligibility counts.

**Inputs and model.** Accepted official Koh Joint ResNet-50 CUB70 Standard seed-1 final-test export, plus released CUB masks or original image-level labels when the figure names them. No new model is fitted; this aggregates earlier exact-concept measurements into coarse anatomical groups.

**Numerical record.** The following code cell prints the complete table behind
the picture, including per-part/per-value denominators and exclusions. The plot
is a visual summary of that table, not a second hidden calculation.

### Figure 11b · How are the available CUB contributors distributed across coarse anatomical groups?

**How to read the figure.** Every panel uses the same coarse-group order: tail, wing, beak, leg, eye,
neck, body, head. Panel A is the positive-label/mapped-mask-absence fraction.
Panel B is `1 - median balanced accuracy` across non-collapsed exact concepts.
Panel C is the median visible-minus-hidden raw-z association. Panel D is the
median hidden-positive-minus-hidden-negative context gap. Panel E is the
standard deviation of species residuals after exact concept and mask state
are centered. Higher means more of the named quantity, but these quantities
have different units and none is a controlled CUB swap failure rate.


In [ ]:
# ALT: Five aligned coarse-group CUB panels showing mask disagreement, concept difficulty, natural visibility association, hidden context separation, and species residual variation without inventing a swap outcome.
CUB_GROUP_ORDER=["tail","wing","beak","leg","eye","neck","body","head"]
collapsed_names=set(HEALTH.loc[HEALTH.collapsed,"concept_name"])
conflict_group=(EXACT.groupby("mask_group").agg(n_hidden=("n_hidden","sum"),n_positive=("n_positive","sum")))
conflict_group["label_mask_absence_rate"]=conflict_group.n_hidden/conflict_group.n_positive.replace(0,np.nan)
difficulty=(HEALTH[(~HEALTH.collapsed)&(HEALTH.n_positive>=10)&(HEALTH.n_negative>=10)].groupby("mask_group").agg(
    median_balanced_accuracy=("balanced_accuracy","median"),n_health_concepts=("concept_name","nunique")))
difficulty["median_classification_error"]=1-difficulty.median_balanced_accuracy
eligible=EXACT[~EXACT.concept_name.isin(collapsed_names)].copy()
vis=(eligible[(eligible.n_visible>=10)&(eligible.n_hidden>=10)].groupby("mask_group")
     .agg(median_visibility_effect=("visibility_effect","median"),n_visibility_concepts=("concept_name","nunique")))
ctx=(eligible[(eligible.n_hidden>=10)&(eligible.n_hidden_negative>=10)].groupby("mask_group")
     .agg(median_context_gap=("context_gap","median"),n_context_concepts=("concept_name","nunique")))
species=(SP.groupby("mask_group").agg(species_residual_sd=("residual","std"),
                                        n_species_cells=("residual","size")))
CUB_SYN=(conflict_group.join(difficulty,how="outer").join(vis,how="outer")
         .join(ctx,how="outer").join(species,how="outer").reindex(CUB_GROUP_ORDER))
panels=[
    ("label_mask_absence_rate","A · DATA CHECK: label / mask absent",(0,1)),
    ("median_classification_error","B · HEALTH CHECK: concept error",(0,1)),
    ("median_visibility_effect","C · LOCAL EVIDENCE: visible − hidden z",None),
    ("median_context_gap","D · CONTEXT: hidden positive − negative z",None),
    ("species_residual_sd","E · SPECIES CONTEXT: residual spread",None),
]
fig,axes=plt.subplots(1,5,figsize=(19,4.8),sharey=True)
colors=[COLORS.get(g,"#888888") for g in CUB_GROUP_ORDER]
for ax,(column,title,limits) in zip(axes,panels):
    ax.barh(np.arange(len(CUB_GROUP_ORDER)),CUB_SYN[column],color=colors)
    if column in ["median_visibility_effect","median_context_gap"]: ax.axvline(0,color="black",lw=.8)
    if limits: ax.set_xlim(*limits)
    ax.set_title(title,fontsize=9); ax.set_yticks(np.arange(len(CUB_GROUP_ORDER)),CUB_GROUP_ORDER)
    ax.invert_yaxis()
fig.suptitle("Figure 11b · CUB observational contributors by coarse anatomical group; no controlled backwash outcome")
plt.tight_layout(); plt.show(); display(CUB_SYN.round(3))
from IPython.display import Markdown
def rank_text(column,ascending=False):
    return " > ".join(CUB_SYN[column].dropna().sort_values(ascending=ascending).index.tolist())
display(Markdown(
    "**How to read this comparison.** Each panel answers a different question and "
    "uses different units. Larger values mean more mask disagreement in A, worse "
    "ordinary concept classification in B, a larger visible-minus-hidden association "
    "in C, more separation without the mapped mask in D, and more species-to-species "
    "variation after concept/visibility centering in E. None is a CUB swap failure rate.\n\n"
    f"**Observed rank orders after coarse grouping.** Mask absence: {rank_text('label_mask_absence_rate')}. "
    f"Concept error: {rank_text('median_classification_error')}. "
    f"Hidden context gap: {rank_text('median_context_gap')}. "
    f"Species residual spread: {rank_text('species_residual_sd')}.\n\n"
    "**Limited conclusion.** Agreement across several panels would identify repeatedly "
    "affected groups. Disagreement means CUB's contributors are distributed rather than "
    "conveniently concentrated in one part. Even agreement cannot create the missing "
    "controlled CUB backwash outcome."
))


### First-pass review slot for Figure 11b

This slot is intentionally **INCOMPLETE** until the figure above has executed
from the accepted Koh Joint manifest and has been displayed in chat.

- **Literal observation:** Fill from the rendered axes and printed denominators;
  do not copy values from the superseded `minimal_cbm` report.
- **Strongest alternative to test:** Coarse-group medians can hide opposite exact-concept effects and the five panels use different units and denominators.
- **Discriminating test:** Return to Figure 11a whenever a group-level bar suggests a common explanation.
- **Verdict:** choose `KEEP`, `REVISE`, `REMOVE`, or `MISSING EVIDENCE` only
  after visual inspection.
- **Limited conclusion:** state only what this output directly supports.
- **Next question if this step is interpretable:** Do selected photographs show physical occlusion or released-mask limitations?


## 12 · Do the numerical extremes correspond to pose, coarse masks, collapse, or contextual prediction?

**Question.** Do the numerical extremes correspond to pose, coarse masks, collapse, or contextual prediction?

**Variables and prediction.** Select cases by declared numerical rules: high conflict/high context gap, high conflict/low gap, strong positive visibility effect, and negative visibility effect. The photograph and all 11 masks must be inspected before assigning an explanation.

**Method.** Display original image, complete mask overlay, exact variables, species, and sample counts.

**Inputs and model.** Accepted official Koh Joint ResNet-50 CUB70 Standard seed-1 final-test export, plus released CUB masks or original image-level labels when the figure names them. No diagnostic is trained. Examples are selected by declared numerical rules from earlier figures, never by visual preference.

**Numerical record.** The following code cell prints the complete table behind
the picture, including per-part/per-value denominators and exclusions. The plot
is a visual summary of that table, not a second hidden calculation.

### Figure 12 · Do the numerical extremes correspond to pose, coarse masks, collapse, or contextual prediction?

**How to read the figure.** Each case occupies two rows: a mapped-mask-absent positive image followed by
a mapped-mask-visible positive image for the same exact concept. Columns are
the photograph, the mapped-region overlay, and every available released-mask
overlay. Titles give species, exact concept, `c`, `c_hat`, raw `z`, and mapped
area; tables give the selection rule and denominators. The images decide
whether an extreme is genuine occlusion, missing/coarse annotation, or
plausible contextual prediction. Collapsed concepts are excluded.


In [ ]:
# ALT: Four rule-selected CUB70 cases, each showing hidden and visible photographs beside overlays of all available released masks and exact raw-logit records.
from PIL import Image
mask_root=CURATED/"cub70"/"masks"/"AnnotationMasksPerclass"
if not mask_root.is_dir(): mask_root=CURATED/"cub70"/"masks"
image_root=CURATED/"CUB_200_2011"/"images"; image_lookup={p.stem:p for p in image_root.rglob("*.jpg")}
collapsed_names=set(HEALTH.loc[HEALTH.collapsed,"concept_name"])
eligible=EXACT[(EXACT.n_visible>=10)&(EXACT.n_hidden>=10)&(EXACT.n_hidden_negative>=10)
    & EXACT.context_gap.notna()&EXACT.visibility_effect.notna()
    & ~EXACT.concept_name.isin(collapsed_names)].copy()
conflict_q75=float(eligible.label_mask_conflict.quantile(.75))
high=eligible[eligible.label_mask_conflict>=conflict_q75]
picks=[("high conflict + high context gap",high.nlargest(1,"context_gap").iloc[0]),
       ("high conflict + low context gap",high.nsmallest(1,"context_gap").iloc[0]),
       ("strong positive visibility effect",eligible.nlargest(1,"visibility_effect").iloc[0]),
       ("negative visibility effect",eligible.nsmallest(1,"visibility_effect").iloc[0])]
mask_colors={p:plt.cm.tab20(i/20) for i,p in enumerate(CUB70_PARTS)}
mapped_parts={"eye":["left_eye","right_eye"],"wing":["left_wing","right_wing"],
              "leg":["left_leg","right_leg"]}
def choose(row,state):
    d=J70[(J70.concept_name==row.concept_name)&(J70.gt_label==1)]
    d=d[d.visible] if state=="visible" else d[~d.visible]
    return d.iloc[(d.z-row.z_visible).abs().argmin()] if len(d) and state=="visible" else (d.iloc[(d.z-row.z_hidden).abs().argmin()] if len(d) else None)
def overlays(stem,group):
    rgb=np.asarray(Image.open(image_lookup[stem]).convert("RGB")); all_ov=rgb.astype(float)/255; mapped_ov=all_ov.copy()
    rr=RAWVIS[RAWVIS.image_name==stem]; cid=int(rr.class_idx.iloc[0])+1; present=[]
    for p in CUB70_PARTS:
        f=mask_root/str(cid)/f"{stem}_{p}.png"
        if not f.exists(): continue
        m=np.asarray(Image.open(f).convert("L"))>0
        if m.shape!=rgb.shape[:2]: m=np.asarray(Image.fromarray(m.astype("uint8")*255).resize((rgb.shape[1],rgb.shape[0]),Image.Resampling.NEAREST))>0
        all_ov[m]=.4*all_ov[m]+.6*np.array(mask_colors[p][:3]); present.append(p)
        if p in mapped_parts.get(group,[group]): mapped_ov[m]=.3*mapped_ov[m]+.7*np.array(mask_colors[p][:3])
    return rgb,mapped_ov,all_ov,present
# Two rows per case keep each photograph large enough to inspect in HTML:
# hidden original/mapped/all masks, then visible original/mapped/all masks.
records=[]; fig,axes=plt.subplots(8,3,figsize=(13,28))
for r,(label,row) in enumerate(picks):
    for offset,state in [(0,"hidden"),(1,"visible")]:
        rr=2*r+offset
        rec=choose(row,state)
        for k in range(3): axes[rr,k].axis("off")
        if rec is None: axes[rr,0].text(.5,.5,"no example",ha="center"); continue
        if rec.concept_name!=row.concept_name: raise RuntimeError("example/concept mismatch")
        rgb,mapped,all_ov,present=overlays(rec.image,row.mask_group)
        axes[rr,0].imshow(rgb); axes[rr,1].imshow(mapped); axes[rr,2].imshow(all_ov)
        axes[rr,0].set_title(f"case {r+1}: {label}\n{state}: {rec.image}; species {rec.y_true}\n{row.concept_name}\nc={int(rec.gt_label)}, c_hat={int(rec.pred_label)}, z={rec.z:.3f}, area={rec.area_frac:.4f}",fontsize=9)
        axes[rr,1].set_title(f"mapped {row.mask_group} mask",fontsize=10)
        axes[rr,2].set_title("all available masks\n"+", ".join(present),fontsize=8)
        records.append({"case":r+1,"rule":label,"state":state,"image":rec.image,"species":rec.y_true,
            "concept_name":row.concept_name,"mask_group":row.mask_group,"c":int(rec.gt_label),
            "c_hat":int(rec.pred_label),"z":rec.z,"area_frac":rec.area_frac,
            "label_mask_conflict":row.label_mask_conflict,"visibility_effect":row.visibility_effect,
            "context_gap":row.context_gap,"n_visible":row.n_visible,"n_hidden":row.n_hidden,
            "n_hidden_negative":row.n_hidden_negative})
fig.suptitle("Figure 12 · Rule-selected photographs and complete mask overlays")
plt.tight_layout(); plt.show()
display(pd.DataFrame([{"selection_rule":label,"conflict_q75_threshold":conflict_q75,**row.to_dict()} for label,row in picks])
    [["selection_rule","conflict_q75_threshold","concept_name","mask_group","label_mask_conflict","visibility_effect","context_gap","n_visible","n_hidden","n_hidden_negative"]].round(3))
display(pd.DataFrame(records).round(3))


### First-pass review slot for Figure 12

This slot is intentionally **INCOMPLETE** until the figure above has executed
from the accepted Koh Joint manifest and has been displayed in chat.

- **Literal observation:** Fill from the rendered axes and printed denominators;
  do not copy values from the superseded `minimal_cbm` report.
- **Strongest alternative to test:** Released masks can be missing or coarser than the named attribute even when a person can see the region, so numerical mask absence is not automatically physical occlusion.
- **Discriminating test:** Inspect all eight photographs and their overlays before interpreting the selected numerical extremes.
- **Verdict:** choose `KEEP`, `REVISE`, `REMOVE`, or `MISSING EVIDENCE` only
  after visual inspection.
- **Limited conclusion:** state only what this output directly supports.
- **Next question if this step is interpretable:** What can be concluded directly across FunnyBird and CUB70?


## 13 · Direct question-matched FunnyBird/CUB evidence table

| Scientific question | FunnyBird operation | CUB operation | Same operation? | Allowed conclusion |
|---|---|---|---|---|
| Are outputs usable? | raw-`z` health on 26 outputs | same health test on 112 outputs | yes | interpret only non-collapsed outputs |
| Do named pixels matter? | controlled `response_delta` after same-scene insertion | natural visible-minus-hidden `z` | no | causal FunnyBird response; observational CUB association |
| Does context remain? | donorward response but old source still wins | hidden-positive minus hidden-negative `z` | no | exact FunnyBird event; observational CUB separation |
| Is matched recall a useful warning? | calibrate recall gaps against controlled exact-value event rates | apply the identical matched-support calculation | same diagnostic, different validation strength | at most a provisional ordinal warning, never a CUB event rate |
| Does visibility contribute? | same-render target area | natural mask state, area, and bilateral count | weaker in CUB | CUB remains pose/species/mask-quality confounded |
| Does exact value matter? | controlled post-swap value confusion | natural exact-concept health and matched species gaps | no | related difficulty question, not equivalent operation |
| Does species organize scores? | exact-pair residual and controlled swap diagnostics | exact-concept/mask-state residual plus held-out prediction | observational in CUB | association can generalize without identifying the causal visual cue |
| Should CUB70 use RLv2? | matched label intervention already tested on FunnyBird | no accepted CUB70 relabel/retrain | unavailable | recommend only as a future causal test if mask conflict is credible and behavior aligns |
| Does the pattern survive 200 species? | not applicable | official full-CUB Koh result is incomplete | unavailable | defer; do not substitute the legacy full-CUB export |

### CUB causal boundary

Notebook 05 may conclude that CUB does or does not show converging
**observational ingredients** of context-dependent concept prediction. It
may not claim a CUB donor/source backwash event because no accepted donor
response exists.


## 14 · Standard-CUB evidence ledger

This source revision deliberately removes all numerical conclusions from
the superseded `minimal_cbm`-CBM render. The following cells must execute
from the official Koh manifest and every current figure must be reviewed
before the status column is finalized.

| Predicate or explanation | Direct measurement | Status before new render review |
|---|---|---|
| population and mask coverage understood | Figure 1 | `INCOMPLETE: OFFICIAL-KOH RENDER/REVIEW` |
| species/concept shortcut available | Figure 2 | `INCOMPLETE: OFFICIAL-KOH RENDER/REVIEW` |
| label/released-mask conflict measured | Figure 3 | `INCOMPLETE: RENDER/IMAGE REVIEW` |
| exact outputs usable | Figure 4 | `INCOMPLETE: OFFICIAL-KOH HEALTH REVIEW` |
| species information recoverable from labels/raw scores | Figure 4b | `INCOMPLETE: OFFICIAL-KOH RENDER/REVIEW` |
| natural visibility association | Figure 5 | `INCOMPLETE: OFFICIAL-KOH RENDER/REVIEW` |
| hidden context separation | Figure 6 | `INCOMPLETE: OFFICIAL-KOH RENDER/REVIEW` |
| bilateral/area alternatives | Figure 7 | `INCOMPLETE: OFFICIAL-KOH RENDER/REVIEW` |
| matched recall and raw-z species gaps | Figure 8 plus Appendix A | `INCOMPLETE: CALIBRATION AND CUB RENDER/REVIEW` |
| concept-level accounting | Figure 9 | `INCOMPLETE: OFFICIAL-KOH RENDER/REVIEW` |
| species residual | Figure 10 | `INCOMPLETE: OFFICIAL-KOH RENDER/REVIEW` |
| row-level accounting | Figure 11 | `INCOMPLETE: OFFICIAL-KOH RENDER/REVIEW` |
| exact and grouped synthesis | Figures 11a–11b | `INCOMPLETE: OFFICIAL-KOH RENDER/REVIEW` |
| visual explanations inspected | Figure 12 | `INCOMPLETE: IMAGE-BY-IMAGE REVIEW` |
| full-CUB robustness | later chapter | `INCOMPLETE: OFFICIAL FULL-CUB MODEL MISSING` |

**Next report question.** Only after this ledger is filled from the new
render may the next notebook ask whether CUB70 MCBM changes the accepted
observational quantities.


# Methods appendix A · Does matched recall track known FunnyBird backwash?

**Question and prediction.** Before treating a CUB70 recall gap as a
warning, test whether the same metric is larger for FunnyBird exact
concepts with more known controlled swap failures. A useful ordinal
warning should rise with the controlled event rate.

**Exact quantities.** For concept `j` and matched species `A,B`:

`recall_gap_jAB = |P(z_j>0 | c_j=1,A) - P(z_j>0 | c_j=1,B)|`.

The concept-level value is the mean across at most 50 eligible species
pairs. The accepted FunnyBird final test contains only ten images per
species, so its fixed small-population rule requires at least two
positive and two negative images in each species. Positive and negative
counts are then matched and bootstrapped. This remains the original
positive-and-negative rule; the all-positive-species fallback is not
used. The setup table also prints what thresholds 1, 2, and 3 would
retain. The controlled target is

`mean(1[response_delta>0 and m_cf<0])`

over swaps that inserted that exact concept.

**Concrete example.** Suppose both species have four positive and two
negative images for one exact value. If the saved concept output is
positive on all four positive images from species A and two of four from
species B, the positive-recall gap is `|4/4-2/4|=0.50`. If eight of 20 swaps inserting
that value move donorward but still finish source-negative, its controlled
event rate is `8/20=0.40`.

**Inputs and model.** Accepted seed-1 Standard FunnyBird Koh Joint
ordinary-image export and its accepted 5,000 fixed swaps. No new
classifier is trained. The model's own `z>0` threshold is reused.

**Axes and decision rule.** Each point below is one exact FunnyBird
concept and color is part. x is its matched species gap. y is its known
controlled event rate. The left panel uses recall; the right panel uses
the label-conditioned raw-score companion. The method earns only
`PROVISIONAL ORDINAL WARNING PROXY` when the recall association is
positive overall, positive after subtracting part means, and positive in
at least four of five leave-one-part-out checks. No image-row error bars
are used as model uncertainty; only more trained seeds can provide that.


In [ ]:
# ALT: Two exact-concept scatter plots calibrating ordinary-image matched species gaps against the known controlled FunnyBird swap event rate, with every point named and part-colored.
FB_CALIBRATION_COLORS=FB_CALIBRATION.part.map(
    {"tail":"#6A0DAD","wing":"#0072B2","beak":"#E69F00",
     "foot":"#009E73","eye":"#CC79A7"})
raw_correlation=FB_CALIBRATION.mean_label_conditioned_raw_z_gap.corr(
    FB_CALIBRATION.controlled_event_rate,method="spearman")
fig,axes=plt.subplots(1,2,figsize=(14,5.5),sharey=True)
panels=[("mean_recall_gap","matched positive-recall gap"),
        ("mean_label_conditioned_raw_z_gap","matched label-conditioned raw-z gap")]
for ax,(column,label) in zip(axes,panels):
    ax.scatter(FB_CALIBRATION[column],FB_CALIBRATION.controlled_event_rate,
               c=FB_CALIBRATION_COLORS,s=52)
    for row in FB_CALIBRATION.itertuples():
        ax.annotate(row.concept_name,(getattr(row,column),row.controlled_event_rate),
                    fontsize=6,xytext=(3,2),textcoords="offset points")
    ax.set_xlabel(label); ax.set_ylabel("controlled event rate for inserted exact value")
    ax.set_ylim(-.03,1.03); ax.grid(alpha=.2)
fig.suptitle("Appendix Figure A1 · Calibration of an ordinary-image warning against controlled FunnyBird swaps")
plt.tight_layout(); plt.show()
display(FB_CALIBRATION[["part","concept_name","n_species_pairs","mean_recall_gap",
    "mean_balanced_accuracy_gap","mean_label_conditioned_raw_z_gap","swap_rows",
    "controlled_event_rate","donor_win_rate","median_response_delta","median_final_margin"]].round(3))
display(FB_CALIBRATION_CHECKS.round(3))
display(pd.DataFrame([{"recall_proxy_verdict":FB_PROXY_VERDICT,
    "raw_z_gap_vs_controlled_event_spearman":raw_correlation,
    "eligible_exact_concepts":len(FB_CALIBRATION),
    "species_pairs":len(FB_RECALL_PAIRS),"training":False}]).round(3))
from IPython.display import Markdown
overall=float(FB_CALIBRATION_CHECKS.loc[FB_CALIBRATION_CHECKS.check=="all exact concepts",
              "spearman_recall_vs_controlled_event"].iloc[0])
centred=float(FB_CALIBRATION_CHECKS.loc[FB_CALIBRATION_CHECKS.check=="within-part centred",
              "spearman_recall_vs_controlled_event"].iloc[0])
positive_loo=int((FB_CALIBRATION_CHECKS.loc[FB_CALIBRATION_CHECKS.check=="leave one part out",
                 "spearman_recall_vs_controlled_event"]>0).sum())
display(Markdown(f'''
### Appendix Figure A1 result

- **Literal result:** recall-gap versus controlled-event Spearman is
  `{overall:.3f}` overall and `{centred:.3f}` after subtracting each
  part's mean. `{positive_loo}` of 5 leave-one-part-out correlations are
  positive. The label-conditioned raw-score companion correlation is
  `{raw_correlation:.3f}`.
- **What it supports:** `{FB_PROXY_VERDICT}`.
- **Plausible alternative:** any overall association can be created by
  part identity, small species cells, or one seed rather than a portable
  concept-level warning.
- **What distinguishes it:** the within-part and omitted-part checks above,
  followed by independent trained seeds.
- **Limited conclusion:** even a passing result ranks warning signs only;
  it does not estimate a CUB backwash rate and cannot replace a controlled
  part swap.
- **Next question:** apply the same formula to CUB70 Figure 8 using its
  declared three-per-label support minimum, with this verdict printed
  beside it.
'''))


# Methods appendix B · CUB edit proxies not used in the main claim

These completed attempts are preserved because they delimit what CUB's
available masks can support:

1. **Reciprocal whole-part deletion:** `METHOD NOT CALIBRATED FOR
   CROSS-DATASET CAUSAL COMPARISON`. The shared edit did not reproduce the
   clean FunnyBird deletion and sometimes damaged meaningful control regions.
2. **Randomized patch masking V1/V2:** selected examples supported local
   pixel response, but the all-part calibration and wing coverage were not
   sufficient for a population-level cross-dataset claim.
3. **Beak/tail paste pilot:** `VALID TEST, NO SUPPORT FOR POSITIVE DONOR
   RESPONSE`. Therefore its negative final margins cannot be interpreted as
   retained-source backwash.

These outcomes reject the proposed edit measurements for their intended
causal use. They do not reject the observational analyses in Figures 1–12
and do not weaken the validated FunnyBird renderer swap.

Full code and artifacts remain in the repository and `CURATED_DATA`; this
report does not rerun them.


# Provenance appendix

The table below records the live Git commit, prediction and mask inputs,
SHA-256 hashes, population counts, collapse tolerance, and exclusions.


In [ ]:
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for block in iter(lambda:f.read(1024*1024),b""): h.update(block)
    return h.hexdigest()
commit=subprocess.run(["git","rev-parse","HEAD"],cwd=REPO,capture_output=True,text=True,check=True).stdout.strip()
prov=[]
for role,path in [("official Koh CUB70 manifest",CUB70_MANIFEST),
                  ("official Koh CUB70 prediction export",E70P),
                  ("visibility parquet",VIS),
                  ("FunnyBird calibration manifest",FB_MODEL_MANIFEST),
                  ("FunnyBird controlled-swap manifest",FB_SWAP_MANIFEST)]:
    prov.append({"role":role,"path":str(path),"sha256":sha256_file(path)})
display(pd.DataFrame(prov)); display(pd.DataFrame([{"git_commit":commit,"seed":1,
    "training_protocol":cub70_meta.get("training_protocol"),
    "target_epochs":cub70_meta.get("target_epochs"),
    "prediction_images":E70.image.nunique(),"mask_matched_images":J70.image.nunique(),
    "species":E70.y_true.nunique(),"exact_concepts":E70.concept_name.nunique(),
    "collapsed_concepts":int(HEALTH.collapsed.sum()),"collapse_tolerance":COLLAPSE_TOL,
    "visibility_threshold_area_fraction":.001}]))
